# ArchiCheck — Extracción Geométrica (Plantilla)

**Usar para cada proyecto nuevo. Solo editar la Celda 3.**

1. Celda 1 — instala dependencias base
2. Celda 2 — sube el PDF del proyecto
3. **Celda 3 — configura páginas y escalas** ← única celda que editas
4. Celda 4 — OpenCV: recintos, áreas, anchos
5. **Celda 4b — carga Grounding DINO + SAM 2** ← ejecutar UNA VEZ por sesión (requiere GPU T4)
6. Celda 4c — detección semántica de elementos (puertas, ventanas, escaleras, rampas)
7. Celda 5 — visualización con detecciones superpuestas
8. Celda 6 — informe consola + guardar JSON
9. Celda 7 — descargar resultados

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 1 — Instalar dependencias (~60 segundos)
# ══════════════════════════════════════════════════════════
print('Instalando librerías...')
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pymupdf', 'opencv-python-headless', 'matplotlib',
                'Pillow', 'numpy', 'requests'], check=True)
print('✓ Listo')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 2 — Subir PDF, validar que sea vectorizado y convertir a imágenes
# ══════════════════════════════════════════════════════════
import fitz
import numpy as np
import cv2
from google.colab import files

print('Selecciona el PDF del plano cuando aparezca el botón:')
uploaded  = files.upload()
pdf_name  = list(uploaded.keys())[0]
pdf_bytes = uploaded[pdf_name]

ZOOM = 3
DPI  = 72 * ZOOM

doc = fitz.open(stream=pdf_bytes, filetype='pdf')
print(f'\nPDF: "{pdf_name}" — {len(doc)} página(s)')

# ── VALIDACIÓN: el PDF debe ser vectorizado, no un escaneo ─────
# FIX 2026-07-23: se detecto que el PDF de prueba (Plaza Pedro de Valdivia) es
# vectorizado — 248 items de texto real con coordenadas exactas y 3722 trazos
# vectoriales en una sola pagina. Eso abre la puerta a extraer cotas y lineas
# de muro/puerta/ventana como datos exactos en vez de adivinar desde pixeles.
# Pero esto SOLO funciona si el PDF es vectorizado. Un escaneo o foto del plano
# impreso no tiene texto ni trazos extraibles — hay que rechazarlo ahora mismo,
# antes de gastar tiempo de GPU/API en un archivo que no va a rendir bien.
print('\nValidando que el PDF sea vectorizado (no un escaneo)...')
total_text_len  = 0
total_drawings  = 0
for _pg in doc:
    total_text_len += len(_pg.get_text('text').strip())
    total_drawings += len(_pg.get_drawings())

print(f'  Texto extraído del PDF : {total_text_len} caracteres')
print(f'  Trazos vectoriales     : {total_drawings}')

MIN_TEXT_CHARS = 50
MIN_DRAWINGS   = 20
if total_text_len < MIN_TEXT_CHARS or total_drawings < MIN_DRAWINGS:
    raise ValueError(
        "\n⛔  Este PDF parece ser un ESCANEO o imagen rasterizada, no un PDF vectorizado.\n"
        f"    Texto extraído: {total_text_len} caracteres (mínimo {MIN_TEXT_CHARS})\n"
        f"    Trazos vectoriales: {total_drawings} (mínimo {MIN_DRAWINGS})\n\n"
        "    Por ahora ArchiCheck solo procesa PDF vectorizados, exportados directo\n"
        "    desde el software de diseño (AutoCAD, Revit, ArchiCAD, etc. → 'Exportar a PDF'),\n"
        "    no un escaneo, foto o PDF impreso-y-vuelto-a-escanear del plano.\n"
        "    Pedile al arquitecto el PDF vectorizado original y volvé a subirlo.\n"
    )
print('  ✓ PDF vectorizado confirmado — continuando.\n')

paginas = []
for i, page in enumerate(doc):
    mat     = fitz.Matrix(ZOOM, ZOOM)
    pix     = page.get_pixmap(matrix=mat, alpha=False)
    buf     = np.frombuffer(pix.tobytes('png'), np.uint8)
    img     = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    paginas.append(img_rgb)
    print(f'  Página {i+1}: {img_rgb.shape[1]}x{img_rgb.shape[0]} px')

print(f'\n✓ {len(paginas)} página(s) cargadas')
print('Revisa los números de página arriba y configura la Celda 3.')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 3 — CONFIGURACIÓN  ← EDITA AQUÍ ANTES DE EJECUTAR
# ══════════════════════════════════════════════════════════
import re, matplotlib.pyplot as plt, matplotlib.patches as patches_plt
from datetime import datetime

# ── NOMBRE DEL PROYECTO ───────────────────────────────────
# Nombre corto del cliente o proyecto. Sin espacios.
# Usa solo letras, números y guiones bajos.
# Ejemplos: 'pdv', 'beaucheff', 'casa_garcia', 'edificio_centro'

NOMBRE_PROYECTO = ''   # ← COMPLETAR

# ── PÁGINAS Y ESCALAS ─────────────────────────────────────
#
# Formato básico:   (numero_pagina, 'escala')
# Con recorte:      (numero_pagina, 'escala', (x1, y1, x2, y2))
#
# x1, y1, x2, y2 son fracciones 0.0–1.0 de la imagen completa:
#   (0.0, 0.0, 1.0, 1.0)  → toda la página  (igual a no poner recorte)
#   (0.0, 0.0, 0.5, 1.0)  → mitad izquierda
#   (0.5, 0.0, 1.0, 0.5)  → cuadrante superior derecho
#
# Escalas comunes: '1:25'  '1:50'  '1:75'  '1:100'  '1:200'  '1:500'
#
# IMPORTANTE: incluye solo secciones con recintos cerrados (plantas).
# Si una página tiene planta + elevación, recorta solo la zona de planta.
# Omite páginas de carátula, especificaciones, elevaciones o detalles sin recintos.
#
# Ejemplos:
#   (2, '1:50')                           → página 2 completa, escala 1:50
#   (3, '1:50', (0.0, 0.0, 0.55, 1.0))   → mitad izquierda de página 3 (planta)
#   (1, '1:100', (0.0, 0.0, 1.0, 0.5))   → mitad superior de página 1

PAGINAS_Y_ESCALAS = [
    # (N°, 'escala'),
    # (N°, 'escala', (x1, y1, x2, y2)),
]

# ── CUADRO DE SUPERFICIES (opcional, 2026-07-31) ──────────
#
# Si el PDF trae una lámina con el cuadro de superficies impreso (tabla
# de áreas por recinto/polígono), indica aquí su número de página para
# que el pipeline extraiga el texto exacto (via PyMuPDF, sin adivinar
# desde la imagen) y lo use para validar las áreas medidas contra el
# valor declarado. Deja en None si el plano no trae un cuadro así, o si
# todavía no quieres usar esta funcionalidad.
#
# Ejemplo: PAGINA_CUADRO_SUPERFICIES = 1

PAGINA_CUADRO_SUPERFICIES = None   # ← opcional, número de página o None

# ─────────────────────────────────────────────────────────
# No edites nada bajo esta línea

# Validar nombre de proyecto
if not NOMBRE_PROYECTO.strip():
    raise ValueError(
        "\n⛔  NOMBRE_PROYECTO está vacío.\n"
        "    Completa la variable antes de continuar. Ejemplo: NOMBRE_PROYECTO = 'pdv'\n"
    )

_slug = re.sub(r'[^a-z0-9]+', '_', NOMBRE_PROYECTO.strip().lower()).strip('_') or 'proyecto'
MESES_ES = ['ene','feb','mar','abr','may','jun','jul','ago','sep','oct','nov','dic']
_now      = datetime.now()
TIMESTAMP = f'{_now.day:02d}{MESES_ES[_now.month-1]}_{_now.strftime("%H%M")}'
BASENAME  = f'archicheck_geometrico_{_slug}_{TIMESTAMP}'

print(f'  Proyecto  : {NOMBRE_PROYECTO}')
print(f'  Timestamp : {TIMESTAMP}')
print(f'  Nombre base de archivos: {BASENAME}')

if not PAGINAS_Y_ESCALAS:
    print('\n⚠ PAGINAS_Y_ESCALAS está vacía — agrega al menos una página antes de continuar.')
else:
    entries = [(e[0], e[1], e[2] if len(e) > 2 else None) for e in PAGINAS_Y_ESCALAS]
    n_pags  = len(entries)
    fig, axes = plt.subplots(1, n_pags, figsize=(14 * n_pags, 9))
    if n_pags == 1:
        axes = [axes]
    for ax, (pag, esc, crop) in zip(axes, entries):
        if pag < 1 or pag > len(paginas):
            ax.set_title(f'Página {pag} — NO EXISTE en el PDF', color='red')
            ax.axis('off')
            continue
        plano_full = paginas[pag - 1]
        h_f, w_f   = plano_full.shape[:2]
        scale_ratio = int(esc.split(':')[1])
        MPX   = 0.0254 * scale_ratio / DPI
        ax.imshow(plano_full)
        if crop:
            x1f, y1f, x2f, y2f = crop
            rect = patches_plt.Rectangle(
                (x1f * w_f, y1f * h_f),
                (x2f - x1f) * w_f, (y2f - y1f) * h_f,
                linewidth=3, edgecolor='#E74C3C', facecolor='#E74C3C22'
            )
            ax.add_patch(rect)
            crop_txt = f'  recorte ({x1f:.0%},{y1f:.0%})→({x2f:.0%},{y2f:.0%})'
            h_crop = int((y2f - y1f) * h_f)
            w_crop = int((x2f - x1f) * w_f)
        else:
            crop_txt = '  sin recorte'
            h_crop, w_crop = h_f, w_f
        ax.set_title(
            f'Página {pag} — escala {esc}\n'
            f'{w_crop}x{h_crop} px analizados  |  {MPX:.5f} m/px{crop_txt}',
            fontsize=10
        )
        ax.axis('off')
    plt.suptitle(
        f'{NOMBRE_PROYECTO} — vista previa páginas a analizar (zona roja = recorte activo)',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()
    print(f'✓ {n_pags} entrada(s) configurada(s). Si el preview es correcto, ejecuta la Celda 4.')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 4 — PROCESAR TODAS LAS PÁGINAS
#   Por cada página: Claude Vision + extracción vectorial + OpenCV + cruce
# ══════════════════════════════════════════════════════════
import base64, json, requests, re, math
import cv2, numpy as np
from datetime import datetime

WORKER_URL = 'https://archicheck-worker.nestragues.workers.dev'

# FIX 2026-07-26 (auditoria completa contra oguc_articulos.json, fuente verificada):
#   - 'pasillo' y 'escalera' citaban el articulo EQUIVOCADO uno del otro: 4.2.2 es
#     "escaleras_minimos" (no pasillos) y 4.2.4 es "carga_ocupacion" (tabla de
#     personas/m2, no ancho de escalera). El ancho minimo real de escalera de uso
#     comun (1.20 m) esta en 4.2.2; el ancho minimo real de pasillo/corredor de uso
#     comun (1.20 m) esta en 4.2.5 ("ancho_vias_evacuacion": "...el ancho minimo de
#     corredores de uso comun es 1,20 m"). Los VALORES (1.20 m ambos) eran correctos,
#     solo la cita estaba cruzada -- ya corregido abajo.
#   - Los minimos de AREA (dormitorio/sala/living/comedor/cocina/bano) citaban Art.
#     4.1.7 OGUC, que verificamos es INTEGRAMENTE sobre accesibilidad universal (ruta
#     accesible, rampas, puertas, ascensores, banos accesibles) -- no contiene NINGUN
#     minimo de superficie por tipo de recinto. Se revisaron ademas 4.1.1/4.1.2/4.1.3/
#     4.5.7 (los unicos otros articulos cargados que mencionan dormitorio/sala/living)
#     y tampoco fijan m2 minimos -- solo alturas, ventilacion e iluminacion. No se
#     encontro en ninguna fuente cargada una base real para estos 6 valores de area;
#     mismo patron de error que la formula de rampa fabricada por Revi (ver roadmap).
#   - FIX 2026-07-26 (b): investigado DS49 (Fondo Solidario de Eleccion de Vivienda,
#     2011, Cuadro Normativo Abreviado MINVU) como candidato. CONFIRMADO QUE NO ES LA
#     FUENTE: (1) los valores no coinciden -- DS49 exige Estar+Comedor COMBINADO 9.40 m2
#     (no separa 'sala'/'living' 10.0 de 'comedor' 8.0 como hace este dict), Dormitorio
#     Principal 7.20 m2 / Segundo Dormitorio 7.00 m2 (no un unico 'dormitorio' 8.0),
#     Cocina 4.00-5.00 m2 (mas exigente que nuestro 3.0, o sea nuestro umbral dejaria
#     pasar cocinas que DS49 rechazaria), Bano 2.50-3.50 m2 (mas exigente que nuestro
#     1.5, mismo problema). (2) Aunque coincidieran, DS49 SOLO aplica a proyectos del
#     programa de vivienda social subsidiada -- no es una norma general OGUC, no aplica
#     a un restaurante ni a vivienda de mercado. Conclusion: estos 6 valores no tienen
#     fuente identificada, ni en OGUC ni en DS49 -- se mantienen SIN VERIFICAR.
#     Se dejan ACTIVOS pero marcados como SIN VERIFICAR -- no se inventa una cita ni
#     se borra el chequeo, se marca la incertidumbre (mismo criterio que
#     'sin_nombre_confirmar' para recintos). Pendiente: decidir si se retiran del motor
#     de reglas o se dejan solo como referencia no vinculante mientras no haya fuente.
OGUC_REGLAS = {
    'dormitorio': (8.0,  None, 'SIN VERIFICAR — sin base confirmada en OGUC (revisar antes de confiar)'),
    'sala'      : (10.0, None, 'SIN VERIFICAR — sin base confirmada en OGUC (revisar antes de confiar)'),
    'living'    : (10.0, None, 'SIN VERIFICAR — sin base confirmada en OGUC (revisar antes de confiar)'),
    'comedor'   : (8.0,  None, 'SIN VERIFICAR — sin base confirmada en OGUC (revisar antes de confiar)'),
    'cocina'    : (3.0,  None, 'SIN VERIFICAR — sin base confirmada en OGUC (revisar antes de confiar)'),
    'bano'      : (1.5,  None, 'SIN VERIFICAR — sin base confirmada en OGUC (revisar antes de confiar)'),
    # FIX 2026-07-26 (d) -- auditado 'pasillo' y 'escalera' contra oguc_pdf.json
    # (extraccion completa, 770 articulos). Confirmado que 4.2.1/4.2.3/4.2.4 SI
    # coinciden textualmente con lo que ya teniamos -- la numeracion no esta rota
    # en general. Pero:
    #   - 'escalera' citaba Art. 4.2.2 -- FALSO. El Art. 4.2.2 real es sobre
    #     "cambio de destino" (autorizacion, informe de profesional), no tiene nada
    #     que ver con escaleras. El articulo real es 4.2.10: "La cantidad y ancho
    #     minimo requerido para las escaleras que forman parte de una via de
    #     evacuacion, conforme a la carga de ocupacion del area servida" -- es una
    #     TABLA por carga de ocupacion (hasta 50 personas: 1,10 m; 51-100: 1,20 m;
    #     101-150: 1,30 m; 151-200: 1,40 m; 201-250: 1,50 m; sobre 250 se exigen 2
    #     escaleras), NO un valor fijo de 1,20 m. No calculamos carga de ocupacion
    #     todavia (requeriria area servida x factor m2/persona del Art. 4.2.4) --
    #     se usa 1,10 m como PISO conservador (el minimo de la tabla, aplica
    #     siempre sin importar ocupacion) en vez de 1,20 m: asi solo se marca
    #     incumplimiento cuando es inequivocamente insuficiente para cualquier
    #     ocupacion, sin arriesgar falsos positivos contra escaleras que si
    #     cumplen para su carga real (que hoy no medimos). Pendiente: implementar
    #     carga de ocupacion real para aplicar la tabla completa.
    #   - 'pasillo' cita Art. 4.2.5 -- el articulo SI es el correcto en tema (el
    #     texto real confirma que el ancho de vias de evacuacion, exceptuando
    #     escaleras, se determina "en base a la carga de ocupacion de la
    #     superficie que sirve"), pero el valor especifico "1,20 m para corredores
    #     de uso comun" que veniamos usando NO aparece textualmente en el articulo
    #     -- no se encontro en esta pasada la tabla equivalente a la de escaleras
    #     (4.2.10) para pasillos/corredores generales. Se mantiene el valor por
    #     ahora (es un minimo de uso muy extendido en la practica) pero queda
    #     marcado como parcialmente verificado, no confirmado al 100%.
    'pasillo'   : (None, 1.20, 'Art. 4.2.5 OGUC — ancho min corredores de uso comun 1,20 m (cita y tema confirmados; valor exacto no verificado al 100% -- ver comentario)'),
    'escalera'  : (None, 1.10, 'Art. 4.2.10 OGUC — tabla por carga de ocupacion, 1,10 m es el piso minimo (hasta 50 personas); puede exigir hasta 1,50 m o 2 escaleras segun ocupacion, no calculado todavia'),
    # FIX 2026-07-26 (c) -- CORRECCION IMPORTANTE tras auditar contra oguc_pdf.json
    # (extraccion completa del PDF oficial, 770 articulos, distinta de la fuente
    # curada oguc_articulos.json que se uso para el fix anterior). El Art. 4.1.7 N°2
    # real es MUCHO mas largo y matizado que el resumen que teniamos: el ancho de
    # rampa NO es un valor fijo de 1,50 m para todas las rampas -- el texto real dice
    # "su ancho debera corresponder a la via de evacuacion que enfrenta o de la que
    # es parte" (variable, 1,10-1,50 m segun el punto 1 de este mismo articulo,
    # 1,50 m especificamente para rutas que conducen a recintos con atencion de
    # publico) Y "las rampas que NO pertenezcan a esas vias del edificio podran
    # tener un ancho minimo de 0,90 m". No hay forma de saber desde la geometria
    # sola si una rampa es "parte de la ruta obligatoria" o no. Se mantiene 1,50 m
    # como default porque el caso de prueba (rampa de acceso a un restaurante,
    # recinto con atencion de publico) cae en ese supuesto -- pero es una
    # simplificacion, no la regla general. Ver tambien el fix de PENDIENTE mas abajo
    # en el bloque "if tipo == 'rampa':", que SI se corrigio a la formula real.
    'rampa'     : (None, 1.50, 'Art. 4.1.7 N°2 OGUC — ancho min 1,50 m (supone ruta a recinto con atencion de publico; puede ser 0,90-1,50 m segun el caso, ver comentario)'),
}

def mejorar_contraste_nitidez(img_rgb):
    """
    CAMBIO 2026-07-23: se descarto la hipotesis de que el 0% de deteccion de
    ventanas fuera un problema de prompt (se probo en vivo, ver roadmap P1) —
    pero el contraste/nitidez de la imagen sigue siendo sospechoso: los
    simbolos de ventana son lineas finas dentro de un vano de muro, y el
    relleno de color de los recintos reduce el contraste justo ahi.
    Esta funcion NO reemplaza la imagen original — genera una segunda version
    con CLAHE (contraste adaptativo) + nitidez, que se manda como imagen
    adicional a Claude, no en vez de la original.
    """
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    contraste = clahe.apply(gray)
    kernel_nitidez = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    nitido = cv2.filter2D(contraste, -1, kernel_nitidez)
    return cv2.cvtColor(nitido, cv2.COLOR_GRAY2RGB)

def _distancia(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

def _agrupar_segmentos(segmentos, radio_busqueda_px, criterio_fn):
    """
    NUEVO 2026-07-24 (reemplaza el enfoque por ancho/patron de guiones de
    path['dashes'], ambos descartados con datos reales — ver roadmap P1,
    diagnostico 2026-07-24: el ancho de linea de un muro y de un simbolo
    tienen la MISMA distribucion en este PDF, y 'dashes' siempre da solido).

    Agrupa indices de 'segmentos' (cada uno con 'p1','p2' en px) en
    componentes conexas via Union-Find, usando criterio_fn(s1, s2) -> bool
    para decidir si dos segmentos se conectan. radio_busqueda_px acota la
    busqueda de candidatos con un bucketing espacial simple, para no
    comparar los ~3000 segmentos de una pagina entre si (O(n^2) es
    demasiado lento en Python puro).
    """
    n = len(segmentos)
    parent = list(range(n))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    def union(i, j):
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[ri] = rj

    cell = max(1, radio_busqueda_px)
    buckets = {}
    def claves(s):
        xs = [s['p1'][0], s['p2'][0]]; ys = [s['p1'][1], s['p2'][1]]
        kx0, kx1 = int((min(xs) - radio_busqueda_px) // cell), int((max(xs) + radio_busqueda_px) // cell)
        ky0, ky1 = int((min(ys) - radio_busqueda_px) // cell), int((max(ys) + radio_busqueda_px) // cell)
        for kx in range(kx0, kx1 + 1):
            for ky in range(ky0, ky1 + 1):
                yield (kx, ky)

    for i, s in enumerate(segmentos):
        for k in claves(s):
            buckets.setdefault(k, []).append(i)

    evaluados = set()
    for i, s in enumerate(segmentos):
        candidatos = set()
        for k in claves(s):
            candidatos.update(buckets.get(k, []))
        for j in candidatos:
            if j <= i or find(i) == find(j):
                continue
            par = (i, j)
            if par in evaluados:
                continue
            evaluados.add(par)
            if criterio_fn(s, segmentos[j]):
                union(i, j)

    grupos = {}
    for i in range(n):
        r = find(i)
        grupos.setdefault(r, []).append(i)
    return list(grupos.values())

def _span_grupo(segmentos, indices):
    """Diagonal del bounding box de todos los extremos del grupo — mide qué
    tan 'extendido' esta la cadena de segmentos conectados, no la cantidad."""
    xs, ys = [], []
    for i in indices:
        xs += [segmentos[i]['p1'][0], segmentos[i]['p2'][0]]
        ys += [segmentos[i]['p1'][1], segmentos[i]['p2'][1]]
    return math.hypot(max(xs) - min(xs), max(ys) - min(ys))

def _angulo_segmento(s):
    dx = s['p2'][0] - s['p1'][0]
    dy = s['p2'][1] - s['p1'][1]
    return math.degrees(math.atan2(dy, dx)) % 180

def extraer_datos_vectoriales(pdf_page, zoom, mpx, crop_px=None, max_largo_trazo_m=3.0):
    """
    Extrae texto y trazos vectoriales directamente del PDF (objeto fitz.Page),
    en vez de adivinarlos desde pixeles. Convierte las coordenadas al mismo
    espacio de pixeles que usa el resto del pipeline (aplicando ZOOM), y
    recorta a crop_px = (x1,y1,x2,y2) en pixeles si se especifica.

    REDISEÑO 2026-07-24 — clasifica cada segmento tipo 'l' en 3 categorias
    por CONECTIVIDAD y GEOMETRIA, no por largo ni ancho de linea individual
    (ambos descartados empiricamente, ver nota arriba):
      1. MURO (protegido, nunca se borra): el segmento pertenece a una
         componente conexa (segmentos que comparten extremo, como una
         cadena de tramos de muro doblando en esquinas) cuyo span total
         supera ~1.5m. Un tramo corto de esquina se protege igual porque
         esta conectado a una cadena larga — esto es lo que faltaba y
         causo la regresion del intento anterior (borrar por largo
         individual punzaba esquinas de muro real).
      2. LINEA DISCONTINUA (se borra): segmentos NO protegidos como muro,
         agrupados por colinealidad + gap regular (patron de guiones
         dibujado a mano con trazos cortos separados, ya que el atributo
         path['dashes'] no sirve en este PDF — siempre da solido).
      3. TRAZO/SIMBOLO (se borra, candidato a puerta/ventana/artefacto):
         el resto — segmentos cortos aislados, curvas bezier, rectangulos
         chicos (iconos de sanitarios, arcos de puerta, etc.)

    Retorna:
      'cotas_texto'         : [{'texto','x','y','w','h'}, ...] — reemplaza el OCR
      'trazos'               : [{'tipo':'l'|'c'|'re'|'qu','puntos':[(x,y),...],'ancho_linea'}, ...]
      'lineas_discontinuas'  : [{'puntos':[(x,y),...],'ancho_linea'}, ...]
      'n_texto', 'n_trazos', 'n_muro_protegido', 'n_lineas_discontinuas', 'n_cadenas_discontinuas'
    """
    def to_px(pt):
        # NUEVO (2026-08-03): esta pagina tiene rotation=270 (confirmado con diagnostico) --
        # get_drawings()/get_text() devuelven coordenadas SIN la rotacion de pagina aplicada,
        # mientras que get_pixmap() (Celda 2, el PNG de fondo) si la aplica -- sin este ajuste,
        # todo lo que sale de aca queda rotado ~90/270 grados respecto al PNG.
        p = pt * pdf_page.rotation_matrix
        return (p.x * zoom, p.y * zoom)

    def dentro_crop(x, y):
        if crop_px is None:
            return True
        cx1, cy1, cx2, cy2 = crop_px
        return cx1 <= x <= cx2 and cy1 <= y <= cy2

    def ajustar(x, y):
        if crop_px is None:
            return (x, y)
        cx1, cy1, _, _ = crop_px
        return (x - cx1, y - cy1)

    # ── Texto: cotas y nombres de recintos, con posicion exacta ──
    # Reemplaza la necesidad de OCR (PaddleOCR) para PDF vectorizados.
    cotas_texto = []
    texto_dict = pdf_page.get_text('dict')
    for block in texto_dict.get('blocks', []):
        for line in block.get('lines', []):
            for span in line.get('spans', []):
                texto = span['text'].strip()
                if not texto:
                    continue
                bbox = span['bbox']  # (x0,y0,x1,y1) en puntos PDF, sin rotacion aplicada
                # mismo ajuste que to_px() -- rotar antes de escalar, y recalcular min/max
                # porque una rotacion de 90/270 puede invertir cual esquina es la minima.
                p0 = fitz.Point(bbox[0], bbox[1]) * pdf_page.rotation_matrix
                p1 = fitz.Point(bbox[2], bbox[3]) * pdf_page.rotation_matrix
                x0, y0 = min(p0.x, p1.x) * zoom, min(p0.y, p1.y) * zoom
                x1, y1 = max(p0.x, p1.x) * zoom, max(p0.y, p1.y) * zoom
                cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
                if not dentro_crop(cx, cy):
                    continue
                ax0, ay0 = ajustar(x0, y0)
                cotas_texto.append({
                    'texto': texto,
                    'x': round(ax0), 'y': round(ay0),
                    'w': round(x1 - x0), 'h': round(y1 - y0),
                })

    # ── Paso 1: recolectar todos los segmentos 'l' (linea recta, 2 puntos)
    #    dentro del crop, con su ancho — son los unicos candidatos a "muro
    #    hecho de tramos cortos" o "cadena de guiones". 'c'/'re'/'qu' se
    #    procesan aparte, mas abajo, con reglas propias mas simples.
    segmentos_l = []
    otros_items = []  # (op, puntos_px, ancho_linea) para 'c'/'re'/'qu'
    UMBRAL_MURO_M = 1.5
    UMBRAL_MURO_PX = UMBRAL_MURO_M / mpx if mpx else float('inf')

    for path in pdf_page.get_drawings():
        ancho_linea = path.get('width') or 0
        for item in path.get('items', []):
            op = item[0]
            if op == 'l':
                p1, p2 = to_px(item[1]), to_px(item[2])
                if not (dentro_crop(*p1) or dentro_crop(*p2)):
                    continue
                segmentos_l.append({'p1': p1, 'p2': p2, 'ancho_linea': ancho_linea, 'color': path.get('color'), 'fill': path.get('fill')})
            elif op == 'c':
                pts = [to_px(p) for p in item[1:5]]
                cx = sum(p[0] for p in pts) / len(pts); cy = sum(p[1] for p in pts) / len(pts)
                if dentro_crop(cx, cy):
                    otros_items.append(('c', pts, ancho_linea))
            elif op == 're':
                r = item[1]
                pts = [to_px(r.tl), to_px(r.tr), to_px(r.br), to_px(r.bl)]
                cx = sum(p[0] for p in pts) / len(pts); cy = sum(p[1] for p in pts) / len(pts)
                if dentro_crop(cx, cy):
                    otros_items.append(('re', pts, ancho_linea))
            elif op == 'qu':
                q = item[1]
                pts = [to_px(q.ul), to_px(q.ur), to_px(q.lr), to_px(q.ll)]
                cx = sum(p[0] for p in pts) / len(pts); cy = sum(p[1] for p in pts) / len(pts)
                if dentro_crop(cx, cy):
                    otros_items.append(('qu', pts, ancho_linea))

    # ── Paso 1.5 (NUEVO 2026-07-31): detectar EJES/lineas de referencia
    #    ANTES del Paso 2, sobre TODOS los segmentos (no solo los que
    #    sobrevivan sin proteger). Motivo: un eje dibujado con guiones
    #    cortos se encadena solo por cercania de extremos (Paso 2) y su
    #    span total (varios metros) supera UMBRAL_MURO_M con facilidad --
    #    quedaba protegido como "muro" ANTES de que el Paso 3 (mas abajo)
    #    llegara a evaluar si era un patron de guiones. Confirmado por el
    #    usuario (captura de plano, Isla de Pascua 2026-07-31): las lineas
    #    discontinuas son EJES de grilla estructural y las lineas delgadas
    #    con marcas de cota son COTAS -- ninguna de las dos es geometria
    #    real del edificio y NUNCA deben dividir/limitar un recinto.
    #    Reutiliza EXACTAMENTE los mismos parametros y la misma funcion
    #    _colineal_y_cerca que ya se usaba en el Paso 3 (ahora definidos
    #    aqui, antes, para poder correr esta pasada temprana) -- no se
    #    afina ni se relaja ningun umbral ya probado.
    TOL_DASH_GAP_PX = 40
    TOL_DASH_ANGULO_DEG = 5
    UMBRAL_DASH_M = 1.0
    UMBRAL_DASH_PX = UMBRAL_DASH_M / mpx if mpx else float('inf')
    MIN_SEGMENTOS_CADENA = 4

    def _colineal_y_cerca(s1, s2):
        dif_ang = abs(_angulo_segmento(s1) - _angulo_segmento(s2))
        dif_ang = min(dif_ang, 180 - dif_ang)
        if dif_ang > TOL_DASH_ANGULO_DEG:
            return False
        return (_distancia(s1['p1'], s2['p1']) <= TOL_DASH_GAP_PX or
                _distancia(s1['p1'], s2['p2']) <= TOL_DASH_GAP_PX or
                _distancia(s1['p2'], s2['p1']) <= TOL_DASH_GAP_PX or
                _distancia(s1['p2'], s2['p2']) <= TOL_DASH_GAP_PX)

    grupos_dash_pre = _agrupar_segmentos(segmentos_l, TOL_DASH_GAP_PX, _colineal_y_cerca) if segmentos_l else []
    es_eje_pre = [False] * len(segmentos_l)
    n_eje_pre_detectado = 0
    for grupo in grupos_dash_pre:
        if len(grupo) < MIN_SEGMENTOS_CADENA:
            continue
        span_pre = _span_grupo(segmentos_l, grupo)
        if span_pre < UMBRAL_DASH_PX:
            continue
        largos_pre = [_distancia(segmentos_l[i]['p1'], segmentos_l[i]['p2']) for i in grupo]
        promedio_pre = sum(largos_pre) / len(largos_pre)
        variacion_pre = (max(largos_pre) - min(largos_pre)) / promedio_pre if promedio_pre else 999
        if variacion_pre > 0.9:
            continue
        for i in grupo:
            if not es_eje_pre[i]:
                es_eje_pre[i] = True
                n_eje_pre_detectado += 1

    # ── Paso 1.6 (NUEVO): detectar LINEAS DE COTA (linea solida, a diferencia
    #    del guion de Paso 1.5) por su firma visual mas especifica: (a) un
    #    segmento largo recto ("portador"), (b) 2+ segmentos cortos que lo
    #    cruzan en angulo (~30-60 grados respecto al portador -- la marca
    #    diagonal de cota, confirmada por el usuario con 2 imagenes de
    #    ejemplo: aparece en CADA punto de division de la cadena de cota,
    #    no solo en los extremos), y (c) al menos una marca cercana a un
    #    texto de cota ya extraido con 100% de precision (cotas_texto, PDF
    #    vectorial). Extiende el diagnostico de solo-proximidad ya iniciado
    #    el 31-jul (mas abajo, ahora reutiliza estas mismas funciones en vez
    #    de redefinirlas) sumando la señal de las marcas, que el diagnostico
    #    original no usaba. NO garantiza 100% -- el mismo tipo de trazo
    #    corto y diagonal ya causo falsos positivos/negativos en este
    #    notebook antes (arcos de puerta, achurado) -- verificar visualmente
    #    (recorte-zoom) contra planos reales antes de confiar, mismo criterio
    #    que ya rigio Paso 1.5 y el fix de achurado (revertido cuando fallo).
    UMBRAL_MARCA_M = 0.20
    UMBRAL_MARCA_PX = UMBRAL_MARCA_M / mpx if mpx else 0
    UMBRAL_PORTADOR_M = 0.5
    UMBRAL_PORTADOR_PX = UMBRAL_PORTADOR_M / mpx if mpx else float('inf')
    ANGULO_MARCA_MIN_DEG = 20
    ANGULO_MARCA_MAX_DEG = 70
    TOL_MARCA_CERCA_PX = 15
    TOL_MARCA_A_TEXTO_PX = 60
    MIN_MARCAS_POR_COTA = 2

    _centros_cotas_texto = [(t['x'] + t['w'] / 2, t['y'] + t['h'] / 2) for t in cotas_texto]

    def _dist_min_a_cota_texto(px, py):
        if not _centros_cotas_texto:
            return float('inf')
        return min(math.hypot(px - cx, py - cy) for cx, cy in _centros_cotas_texto)

    def _punto_medio(s):
        return ((s['p1'][0] + s['p2'][0]) / 2, (s['p1'][1] + s['p2'][1]) / 2)

    def _dist_punto_a_segmento(p, s):
        x0, y0 = p; x1, y1 = s['p1']; x2, y2 = s['p2']
        dx, dy = x2 - x1, y2 - y1
        if dx == 0 and dy == 0:
            return math.hypot(x0 - x1, y0 - y1)
        t = max(0, min(1, ((x0 - x1) * dx + (y0 - y1) * dy) / (dx * dx + dy * dy)))
        px_, py_ = x1 + t * dx, y1 + t * dy
        return math.hypot(x0 - px_, y0 - py_)

    es_cota_pre = [False] * len(segmentos_l)
    n_cota_pre_detectado = 0
    _largos_seg = [_distancia(s['p1'], s['p2']) for s in segmentos_l]
    _candidatos_portador = [i for i, largo in enumerate(_largos_seg) if largo >= UMBRAL_PORTADOR_PX]
    _candidatos_marca = [i for i, largo in enumerate(_largos_seg) if 0 < largo <= UMBRAL_MARCA_PX]

    for ip in _candidatos_portador:
        s_portador = segmentos_l[ip]
        ang_portador = _angulo_segmento(s_portador)
        marcas_de_este = []
        for im in _candidatos_marca:
            if im == ip:
                continue
            s_marca = segmentos_l[im]
            dif_ang = abs(_angulo_segmento(s_marca) - ang_portador)
            dif_ang = min(dif_ang, 180 - dif_ang)
            if dif_ang < ANGULO_MARCA_MIN_DEG or dif_ang > ANGULO_MARCA_MAX_DEG:
                continue
            if _dist_punto_a_segmento(_punto_medio(s_marca), s_portador) > TOL_MARCA_CERCA_PX:
                continue
            marcas_de_este.append(im)
        if len(marcas_de_este) < MIN_MARCAS_POR_COTA:
            continue
        if not any(_dist_min_a_cota_texto(*_punto_medio(segmentos_l[im])) <= TOL_MARCA_A_TEXTO_PX for im in marcas_de_este):
            continue
        if not es_cota_pre[ip]:
            es_cota_pre[ip] = True
            n_cota_pre_detectado += 1
        for im in marcas_de_este:
            if not es_cota_pre[im]:
                es_cota_pre[im] = True
                n_cota_pre_detectado += 1

    print(f'  ✓ COTAS (linea): {n_cota_pre_detectado} segmentos identificados como linea/marca de cota (Paso 1.6)')

    # ── Paso 2: proteger como MURO cualquier segmento conectado (comparte
    #    extremo con tolerancia) a una cadena cuyo span total supere
    #    UMBRAL_MURO_M. Un tramo corto de esquina se protege igual porque
    #    esta conectado a la cadena larga del resto del muro.
    # AJUSTE 2026-07-25: la version con TOL_MURO_PX=12 (~7cm) NO resolvio la
    # regresion -- el recinto gigante (~141/155 m2) siguio apareciendo
    # identico tras aplicar el rediseño por conectividad (ver roadmap P1,
    # verif_combinado_pag2-1/2.png). Hipotesis con evidencia indirecta: los
    # huecos reales entre tramos de muro en las uniones/esquinas de este PDF
    # son mas grandes que 12px, asi que esos tramos no se encadenaban entre
    # si y quedaban sin proteger igual que antes. Se sube el margen bastante
    # (12->35px, ~7cm->20cm) porque el riesgo es asimetrico: proteger de mas
    # como muro cuesta poco (un simbolo real no se borra), proteger de menos
    # repite la regresion catastrofica de fusionar exterior+interior.
    # AJUSTE 2026-07-31: ahora, ademas, nunca se protege un segmento ya
    # identificado como eje/guion en el Paso 1.5 -- por mas que su cadena
    # de conectividad supere el span minimo de muro.
    TOL_MURO_PX = 35
    def _tocan(s1, s2):
        return (_distancia(s1['p1'], s2['p1']) <= TOL_MURO_PX or
                _distancia(s1['p1'], s2['p2']) <= TOL_MURO_PX or
                _distancia(s1['p2'], s2['p1']) <= TOL_MURO_PX or
                _distancia(s1['p2'], s2['p2']) <= TOL_MURO_PX)

    grupos_conectividad = _agrupar_segmentos(segmentos_l, TOL_MURO_PX, _tocan) if segmentos_l else []
    protegido = [False] * len(segmentos_l)
    n_muro_protegido = 0
    n_muro_evitado_por_eje = 0
    n_muro_evitado_por_cota = 0
    for grupo in grupos_conectividad:
        if _span_grupo(segmentos_l, grupo) >= UMBRAL_MURO_PX:
            for i in grupo:
                if es_eje_pre[i]:
                    n_muro_evitado_por_eje += 1
                    continue
                if es_cota_pre[i]:
                    n_muro_evitado_por_cota += 1
                    continue
                protegido[i] = True
                n_muro_protegido += 1
    print(f'  ✓ EJES/RASANTES: {n_eje_pre_detectado} segmentos identificados como referencia (Paso 1.5), '
          f'{n_muro_evitado_por_eje} evitados de proteger como muro pese a superar el span minimo')
    print(f'  ✓ COTAS (linea): {n_muro_evitado_por_cota} segmentos adicionales evitados de proteger como muro (Paso 1.6)')

    # ── NUEVO (2026-08-03): exportar los muros reales, en vez de descartarlos ──
    # grupos_conectividad ya separo los tramos en componentes conexas (Paso 2) y
    # 'protegido' ya excluye lo reclasificado como eje/cota (Paso 1.5/1.6) -- esto
    # solo junta esa geometria ya calculada en un formato exportable, no repite
    # ningun calculo. Cada grupo que califico como muro (span >= 1.5m) se exporta
    # como una LISTA DE SEGMENTOS (no una polilinea ordenada) -- un muro real puede
    # ramificarse en una esquina en T o un cruce, y forzar un orden de camino unico
    # sobre un grafo que puede ramificar no tiene una respuesta correcta unica.
    muros_geo = []
    puertas_geo = []
    # NUEVO (2026-08-04, v2): clasificador geometrico de puertas -- reconoce el
    # arco de giro directamente en los datos vectoriales (los mismos segmentos_l
    # ya extraidos para muros_geo), en vez de depender del centroide que adivina
    # Claude Vision. Complementa al clasificador de Claude Vision (puertas_detalle),
    # no lo reemplaza -- ver roadmap.
    # v2 (instruccion explicita del usuario): 'las puertas solo son las que tienen
    # arco, continuo o discontinuo' -- se descartaron los umbrales de v1 (cantidad
    # minima de segmentos, ancho en un rango) porque un arco discontinuo (guiones)
    # puede tener pocos tramos y cualquier tamano; el UNICO criterio real es si la
    # forma es geometricamente un arco de circulo, sin importar tamano ni cuantos
    # tramos lo dibujan. Se ajusta un circulo (metodo algebraico/Kasa, via numpy,
    # ya importado en esta celda) a los puntos del grupo candidato y se mide que
    # tan bien encajan (residual relativo = desviacion estandar de la distancia al
    # centro, dividido por el radio) -- bajo = es un arco real, alto = no lo es.
    MIN_PUNTOS_ARCO = 6  # minimo para que el ajuste sea significativo (3 puntos
    # cualesquiera siempre arman un circulo perfecto sin decir nada sobre si es real)
    TOL_ARCO_RESIDUAL_REL = 0.15  # que tan circular debe ser el ajuste, sin verificar
    MIN_BARRIDO_ARCO_DEG = 20  # un arco real barre bastante mas que esto; una
    # recta mal ajustada a un circulo gigante barre casi nada
    def _ajustar_circulo(puntos):
        if len(puntos) < MIN_PUNTOS_ARCO:
            return None
        xs = np.array([p[0] for p in puntos], dtype=float)
        ys = np.array([p[1] for p in puntos], dtype=float)
        A = np.column_stack([2 * xs, 2 * ys, np.ones(len(xs))])
        b = xs ** 2 + ys ** 2
        try:
            sol, *_r = np.linalg.lstsq(A, b, rcond=None)
        except Exception:
            return None
        cx, cy, c = sol
        r2 = c + cx ** 2 + cy ** 2
        if r2 <= 0:
            return None
        r = math.sqrt(r2)
        dists = np.sqrt((xs - cx) ** 2 + (ys - cy) ** 2)
        residual_relativo = float(np.std(dists) / r) if r > 0 else 1.0
        # NUEVO (2026-08-04, fix falso positivo): una recta es matematicamente el
        # caso limite de un circulo de radio infinito -- puntos casi colineales
        # pueden dar un ajuste con residual bajo (parece 'buen circulo') aunque no
        # sea un arco real. Confirmado con datos reales: PG01/PG02/PG05 resultaron
        # ser lineas rectas (borde de muro, lineas de rampa) con radio gigante
        # (1.9-2.8m) que el residual solo no detecto. El barrido angular si lo
        # distingue: los puntos de una recta, vistos desde un centro lejano, caen
        # todos en un rango de angulo muy chico -- un arco real barre bastante mas.
        angulos = sorted(float(a) for a in (np.degrees(np.arctan2(ys - cy, xs - cx)) % 360))
        gaps = [angulos[i + 1] - angulos[i] for i in range(len(angulos) - 1)]
        gaps.append(360 - angulos[-1] + angulos[0])
        barrido_deg = 360 - max(gaps)
        return (float(cx), float(cy), float(r), residual_relativo, barrido_deg)
    # Tolerancia para re-agrupar candidatos a arco -- mas laxa que TOL_MURO_PX (35px)
    # porque un arco discontinuo tiene huecos entre guiones; reusa TOL_DASH_GAP_PX
    # (40px), la misma tolerancia ya usada para cadenas de linea discontinua.
    # NUEVO (2026-08-04, fix real de achurado): el ajuste de circulo no distingue
    # de forma confiable un arco real de un parche de achurado (confirmado con datos
    # reales: PG01/PG02 de una corrida resultaron ser las rayas del achurado 'Se
    # construye', con barrido 130-136 grados -- el fix anterior de barrido no las
    # agarraba). La señal que SI separa limpio (verificado con datos reales, 98-99%
    # vs 40-86% del resto): un achurado son decenas de trazos CASI TODOS PARALELOS
    # entre si (rayas repetidas a la misma inclinacion); un arco real, aunque tenga
    # algunos segmentos parecidos, nunca tiene esa uniformidad -- cada tramo de una
    # curva real gira progresivamente.
    TOL_PARALELO_DEG = 3
    UMBRAL_FRACCION_PARALELA = 0.90
    # NUEVO (2026-08-04): descarta simbolos tipo 'flecha de pendiente de rampa' --
    # 2 lineas rectas que convergen a un punto en angulo agudo, confirmado con
    # datos reales que se repite en varias rampas del plano. Se distingue de un
    # arco real por el SALTO de direccion: un arco gira progresivo (saltos chicos
    # y parejos entre segmentos consecutivos); una flecha en V tiene 2 direcciones
    # dominantes con un salto brusco entre ellas. Medido con datos reales: la
    # flecha de rampa da salto_max/barrido=2.30, las 12 puertas reales confirmadas
    # dan 0.11-0.51 -- separacion limpia.
    UMBRAL_SALTO_MAX_RELATIVO = 0.8
    def _salto_maximo_relativo(segmentos_grupo):
        angs = sorted(_angulo_segmento(s) % 180 for s in segmentos_grupo)
        n = len(angs)
        if n < 2:
            return 0.0
        gaps = [angs[i + 1] - angs[i] for i in range(n - 1)]
        gaps.append(180 - angs[-1] + angs[0])
        barrido = 180 - max(gaps)
        gaps_internos = gaps[:-1]
        salto_max = max(gaps_internos) if gaps_internos else 0.0
        return salto_max / barrido if barrido > 0 else 0.0
    def _fraccion_paralela(segmentos_grupo):
        angs = [_angulo_segmento(s) for s in segmentos_grupo]
        n = len(angs)
        if n < 2:
            return 0.0
        n_con_paralelo = 0
        for i in range(n):
            for k in range(n):
                if i == k:
                    continue
                d = abs(angs[i] - angs[k])
                d = min(d, 180 - d)
                if d <= TOL_PARALELO_DEG:
                    n_con_paralelo += 1
                    break
        return n_con_paralelo / n
    def _cerca_arco(s1, s2):
        return (_distancia(s1['p1'], s2['p1']) <= TOL_DASH_GAP_PX or
                _distancia(s1['p1'], s2['p2']) <= TOL_DASH_GAP_PX or
                _distancia(s1['p2'], s2['p1']) <= TOL_DASH_GAP_PX or
                _distancia(s1['p2'], s2['p2']) <= TOL_DASH_GAP_PX)
    TOL_EJE_MURO_DEG = 8  # tolerancia angular para considerar un segmento parte de un muro real
    n_muro_excluido_amarillo = 0
    # NUEVO (2026-08-04): excluir por color el achurado 'Se retira' (amarillo) --
    # el 'Se construye' (rojo) SI se mantiene como muro, porque marca un muro real
    # nuevo (parte del diseno propuesto), no algo descartado. Colores reales de
    # ESTE PDF sin verificar todavia -- ver print de diagnostico mas abajo, que
    # imprime los colores realmente vistos para poder ajustar el rango si hace falta.
    def _es_amarillo(seg):
        for c in (seg.get('color'), seg.get('fill')):
            if c and len(c) >= 3 and c[0] > 0.6 and c[1] > 0.6 and c[2] < 0.4:
                return True
        return False
    n_muro_desalineado = 0
    for grupo in grupos_conectividad:
        if _span_grupo(segmentos_l, grupo) < UMBRAL_MURO_PX:
            continue
        segs_protegidos = [i for i in grupo if protegido[i]]
        if not segs_protegidos:
            continue
        # NUEVO (2026-08-04): de los ya protegidos, exportar solo los alineados a un
        # eje (0/90 grados, +-TOL_EJE_MURO_DEG) -- en ESTE plano (rectilineo) los
        # muros reales corren en 0/90; un arco de giro de puerta barre angulos
        # intermedios de forma continua y el achurado corre en diagonal constante --
        # ninguno de los dos pasa este filtro. NO toca 'protegido' (la mascara que
        # separa recintos via OpenCV sigue exactamente igual, sin riesgo de repetir
        # la regresion del 27-jul) -- solo filtra que segmentos se exportan como
        # parte del poligono de muro. Supuesto a revisar en planos NO rectilineos.
        segs_reales = []
        segs_no_muro = []  # candidatos a arco de puerta (excluidos por angulo, no por color)
        for i in segs_protegidos:
            if _es_amarillo(segmentos_l[i]):
                n_muro_excluido_amarillo += 1
                continue
            ang = _angulo_segmento(segmentos_l[i]) % 90
            if min(ang, 90 - ang) <= TOL_EJE_MURO_DEG:
                segs_reales.append(i)
            else:
                segs_no_muro.append(i)
        n_muro_desalineado += len(segs_protegidos) - len(segs_reales)
        if not segs_reales:
            continue
        segmentos_muro = []
        for i in segs_reales:
            p1 = ajustar(*segmentos_l[i]['p1'])
            p2 = ajustar(*segmentos_l[i]['p2'])
            segmentos_muro.append({'p1': [round(p1[0]), round(p1[1])], 'p2': [round(p2[0]), round(p2[1])]})
        anchos_muro = [segmentos_l[i]['ancho_linea'] for i in segs_reales]
        muros_geo.append({
            'id': f'MU{len(muros_geo) + 1:02d}',
            'segmentos': segmentos_muro,
            'largo_total_m': round(sum(_distancia(segmentos_l[i]['p1'], segmentos_l[i]['p2']) for i in segs_reales) * mpx, 2),
            'ancho_linea_prom': round(sum(anchos_muro) / len(anchos_muro), 2) if anchos_muro else 0,
        })
        # Buscar arcos de puerta entre los segmentos que quedaron fuera del muro --
        # re-agrupa SOLO esos candidatos, con tolerancia laxa (TOL_DASH_GAP_PX) para
        # no perder arcos dibujados discontinuos, y confirma cada grupo ajustando un
        # circulo -- si el ajuste es bueno (residual bajo) es un arco real, sin
        # importar su tamano ni cuantos tramos lo forman.
        if segs_no_muro:
            _sub_segmentos = [segmentos_l[i] for i in segs_no_muro]
            _sub_grupos = _agrupar_segmentos(_sub_segmentos, TOL_DASH_GAP_PX, _cerca_arco)
            for _sub_grupo in _sub_grupos:
                _puntos_arco = []
                for j in _sub_grupo:
                    _puntos_arco.append(_sub_segmentos[j]['p1'])
                    _puntos_arco.append(_sub_segmentos[j]['p2'])
                _ajuste = _ajustar_circulo(_puntos_arco)
                if _ajuste is None:
                    continue
                _cx, _cy, _radio_px, _residual, _barrido_deg = _ajuste
                if _residual > TOL_ARCO_RESIDUAL_REL:
                    continue
                if _barrido_deg < MIN_BARRIDO_ARCO_DEG:
                    continue
                _sub_segmentos_grupo = [_sub_segmentos[j] for j in _sub_grupo]
                if _fraccion_paralela(_sub_segmentos_grupo) >= UMBRAL_FRACCION_PARALELA:
                    continue  # achurado: casi todos los tramos son paralelos entre si
                if _salto_maximo_relativo(_sub_segmentos_grupo) > UMBRAL_SALTO_MAX_RELATIVO:
                    continue  # flecha/cuna en V: 2 direcciones con un salto brusco, no un giro progresivo
                _indices_orig = [segs_no_muro[j] for j in _sub_grupo]
                _segmentos_puerta = []
                for i in _indices_orig:
                    p1 = ajustar(*segmentos_l[i]['p1'])
                    p2 = ajustar(*segmentos_l[i]['p2'])
                    _segmentos_puerta.append({'p1': [round(p1[0]), round(p1[1])], 'p2': [round(p2[0]), round(p2[1])]})
                # Puntos de union: hasta 2 (2026-08-04, v3, pedido explicito del
                # usuario) -- una puerta tipicamente esta ENTRE dos tramos de muro,
                # uno a cada lado del vano ('===========--------------========').
                # No se fuerza que existan 2 -- hay muros que no terminan en nada real
                # (ej. una salida sin puerta), asi que se guardan los que se encuentren
                # (0, 1 o 2), sin inventar un segundo lado que no este ahi.
                # AJUSTE (2026-08-04, v4): buscar desde los 2 EXTREMOS reales del arco
                # (el par de puntos mas separados entre si), no desde CUALQUIER punto
                # muestreado -- la version anterior podia enganchar el punto de muro
                # mas cercano a un tramo INTERMEDIO de la curva, mostrando el punto de
                # union sobre la mitad del arco en vez de en su punta (confirmado con
                # capturas reales del usuario, portal).
                _extremo_a, _extremo_b = _puntos_arco[0], _puntos_arco[0]
                _dist_max_arco = -1.0
                for _pa in _puntos_arco:
                    for _pb in _puntos_arco:
                        _d_ext = _distancia(_pa, _pb)
                        if _d_ext > _dist_max_arco:
                            _dist_max_arco = _d_ext
                            _extremo_a, _extremo_b = _pa, _pb
                _puntos_union_px = []
                for _extremo in (_extremo_a, _extremo_b):
                    _mejor_d, _mejor_pm = None, None
                    for i in segs_reales:
                        for pm in (segmentos_l[i]['p1'], segmentos_l[i]['p2']):
                            _d_local = _distancia(pm, _extremo)
                            if _mejor_d is None or _d_local < _mejor_d:
                                _mejor_d, _mejor_pm = _d_local, pm
                    if _mejor_pm is None:
                        continue
                    if any(_distancia(_mejor_pm, _pu) <= TOL_MURO_PX for _pu in _puntos_union_px):
                        continue  # mismo lado ya capturado por el otro extremo, no lo duplica
                    _puntos_union_px.append(_mejor_pm)
                _puntos_union = []
                for _pm in _puntos_union_px:
                    _pu_aj = ajustar(*_pm)
                    _puntos_union.append([round(_pu_aj[0]), round(_pu_aj[1])])
                puertas_geo.append({
                    'id': f'PG{len(puertas_geo) + 1:02d}',
                    'segmentos': _segmentos_puerta,
                    'ancho_estimado_m': round(_radio_px * mpx, 2),
                    'muro_asociado_id': muros_geo[-1]['id'],
                    'puntos_union': _puntos_union,
                })
    # NUEVO (2026-08-04): puertas dibujadas como curva Bezier real ('c'), no como
    # muchos tramos rectos -- el pipeline de arriba (segmentos_l) nunca las ve,
    # van a 'otros_items' y de ahi a 'trazos' (solo se usan para borrar simbolos
    # del raster, nunca se clasifican). Confirmado con un caso real (ver roadmap):
    # una puerta visible en el plano nunca generaba ningun candidato porque su arco
    # es una Bezier cubica de 4 puntos de control, no una cadena de lineas rectas.
    # Criterio (instruccion explicita del usuario, no un umbral de tamano): un arco
    # de giro de puerta es un cuarto de circulo (~90 grados) -- se muestrean puntos
    # a lo largo de la curva y se reusa _ajustar_circulo (misma funcion ya probada
    # arriba) para confirmar que el barrido cae cerca de 90 grados. Una curva de
    # mobiliario (ej. el ovalo de un WC) no tiene esa forma y se descarta aca.
    N_MUESTRAS_BEZIER = 9
    UMBRAL_BARRIDO_BEZIER_MIN = 75
    UMBRAL_BARRIDO_BEZIER_MAX = 105
    # Piso minimo de plausibilidad (2026-08-04, a pedido del usuario) -- NO es un
    # intento de 'confirmar que es puerta' (eso ya lo hace el barrido ~90 grados),
    # es para descartar ruido sub-centimetrico (tornillos, marcas de detalle,
    # circulos decorativos chicos) que puede tener barrido ~90 por pura casualidad
    # geometrica sin ser ningun simbolo arquitectonico real. Confirmado con datos
    # reales: sin este piso, 146/209 y 64/92 candidatos eran ruido de <5cm de radio.
    UMBRAL_RADIO_BEZIER_MIN_M = 0.5
    def _muestrear_bezier_cubica(p0, p1, p2, p3, n=N_MUESTRAS_BEZIER):
        puntos = []
        for i in range(n):
            t = i / (n - 1)
            mt = 1 - t
            x = mt**3*p0[0] + 3*mt**2*t*p1[0] + 3*mt*t**2*p2[0] + t**3*p3[0]
            y = mt**3*p0[1] + 3*mt**2*t*p1[1] + 3*mt*t**2*p2[1] + t**3*p3[1]
            puntos.append((x, y))
        return puntos
    # AJUSTE (2026-08-04, v4): buscar desde los 2 extremos reales de la curva
    # (muestras[0] y muestras[-1], los puntos P0/P3 de la Bezier -- ya son los
    # extremos exactos, no hace falta buscar el par mas separado como en el caso
    # de lineas rectas) -- mismo motivo que el ajuste de arriba: evitar enganchar
    # el punto de muro mas cercano a un tramo intermedio de la curva.
    def _puntos_union_y_muro(extremo_a_aj, extremo_b_aj):
        puntos_union, muro_id = [], None
        for extremo in (extremo_a_aj, extremo_b_aj):
            mejor_d, mejor_pm, mejor_muro_id = None, None, None
            for m in muros_geo:
                for s in m['segmentos']:
                    for pm in (s['p1'], s['p2']):
                        d = _distancia(pm, extremo)
                        if mejor_d is None or d < mejor_d:
                            mejor_d, mejor_pm, mejor_muro_id = d, pm, m['id']
            if mejor_pm is None:
                continue
            if any(_distancia(mejor_pm, pu) <= TOL_MURO_PX for pu in puntos_union):
                continue
            puntos_union.append(mejor_pm)
            if muro_id is None:
                muro_id = mejor_muro_id
        return puntos_union, muro_id
    n_puertas_bezier = 0
    for _op, _pts, _ancho_linea in otros_items:
        if _op != 'c' or len(_pts) != 4:
            continue
        _muestras = _muestrear_bezier_cubica(*_pts)
        _ajuste_bz = _ajustar_circulo(_muestras)
        if _ajuste_bz is None:
            continue
        _cx_bz, _cy_bz, _radio_bz_px, _residual_bz, _barrido_bz = _ajuste_bz
        if _residual_bz > TOL_ARCO_RESIDUAL_REL:
            continue
        if not (UMBRAL_BARRIDO_BEZIER_MIN <= _barrido_bz <= UMBRAL_BARRIDO_BEZIER_MAX):
            continue
        if _radio_bz_px * mpx < UMBRAL_RADIO_BEZIER_MIN_M:
            continue
        _muestras_aj = [ajustar(x, y) for x, y in _muestras]
        _segmentos_bz = []
        for _i in range(len(_muestras_aj) - 1):
            _p1bz = _muestras_aj[_i]
            _p2bz = _muestras_aj[_i + 1]
            _segmentos_bz.append({'p1': [round(_p1bz[0]), round(_p1bz[1])], 'p2': [round(_p2bz[0]), round(_p2bz[1])]})
        _puntos_union_bz, _muro_id_bz = _puntos_union_y_muro(_muestras_aj[0], _muestras_aj[-1])
        n_puertas_bezier += 1
        puertas_geo.append({
            'id': f'PG{len(puertas_geo) + 1:02d}',
            'segmentos': _segmentos_bz,
            'ancho_estimado_m': round(_radio_bz_px * mpx, 2),
            'muro_asociado_id': _muro_id_bz,
            'puntos_union': [[round(p[0]), round(p[1])] for p in _puntos_union_bz],
        })
    print(f'  ✓ Puertas exportadas desde curva Bezier (arco ~90 grados): {n_puertas_bezier}')
    print(f'  ✓ Muros exportados: {len(muros_geo)} (de {sum(1 for g in grupos_conectividad if _span_grupo(segmentos_l, g) >= UMBRAL_MURO_PX)} grupos protegidos, {n_muro_desalineado} segmentos descartados por angulo no-ortogonal, {n_muro_excluido_amarillo} descartados por color amarillo/achurado Se-retira)')
    print(f'  ✓ Puertas exportadas (clasificador geometrico): {len(puertas_geo)}')
    _colores_vistos = {}
    for _s in segmentos_l:
        for _c in (_s.get('color'), _s.get('fill')):
            if _c:
                _k = tuple(round(_x, 2) for _x in _c)
                _colores_vistos[_k] = _colores_vistos.get(_k, 0) + 1
    _top_colores = sorted(_colores_vistos.items(), key=lambda kv: -kv[1])[:8]
    print(f'  DIAGNOSTICO COLOR (para verificar/ajustar el filtro de amarillo): {_top_colores}')

    # ── DIAGNOSTICO TEMPORAL 2026-07-31 (no cambia ningun resultado) ──
    # Mide span y grosor de cada grupo ya protegido como "muro" en el Paso 2,
    # para separar EJES/COTAS de muros reales con evidencia real antes de
    # diseñar un pre-filtro (ver roadmap P1, hallazgo Isla de Pascua 2026-07-31).
    # Hipotesis a verificar: ejes/cotas deberian aparecer como los grupos de
    # mayor span (varios metros, casi todo el ancho/alto de la pagina) y con
    # ancho_linea mas fino que un muro real. Solo imprime -- no toca 'protegido'
    # ni ningun valor que se guarda en el JSON de salida.
    import statistics as _diag_stats
    import random as _diag_random

    # ── DIAGNOSTICO COTAS 2026-07-31 (temporal, no cambia ningun resultado) ──
    # Hipotesis a verificar (a raiz del hallazgo de que EJES funciona pero
    # el resultado visible casi no cambio, ver roadmap P1): un segmento de
    # COTA (linea solida) deberia estar mucho mas cerca de un texto de cota
    # (numero de la cadena de acotacion, cotas_texto ya extraido con
    # precision del PDF) que un tramo de muro real -- un muro puede tener
    # una cota cerca en un punto puntual, pero una linea de cota corre
    # PEGADA a su propio numero en toda su extension. Mide, por grupo, que
    # fraccion de sus segmentos tiene un texto de cota cerca, a 3 umbrales
    # de distancia distintos -- sin comprometerse a uno solo todavia.
    # NOTA (Paso 1.6): _centros_cotas_texto y _dist_min_a_cota_texto ya se
    # definieron mas arriba (Paso 1.6, filtro real de linea de cota) -- no
    # se redefinen aqui, este bloque las reutiliza tal cual.
    UMBRALES_COTA_PX = [30, 60, 100]
    MAX_MUESTRA_POR_GRUPO = 300  # muestrea grupos enormes para no tardar de mas

    _diag_grupos = []
    for grupo in grupos_conectividad:
        span_px = _span_grupo(segmentos_l, grupo)
        if span_px < UMBRAL_MURO_PX:
            continue
        xs, ys = [], []
        for i in grupo:
            xs += [segmentos_l[i]["p1"][0], segmentos_l[i]["p2"][0]]
            ys += [segmentos_l[i]["p1"][1], segmentos_l[i]["p2"][1]]
        anchos = [segmentos_l[i]["ancho_linea"] for i in grupo]

        muestra = grupo if len(grupo) <= MAX_MUESTRA_POR_GRUPO else _diag_random.sample(grupo, MAX_MUESTRA_POR_GRUPO)
        conteo_umbral = {u: 0 for u in UMBRALES_COTA_PX}
        for i in muestra:
            s = segmentos_l[i]
            mx = (s['p1'][0] + s['p2'][0]) / 2
            my = (s['p1'][1] + s['p2'][1]) / 2
            d = _dist_min_a_cota_texto(mx, my)
            for u in UMBRALES_COTA_PX:
                if d <= u:
                    conteo_umbral[u] += 1
        n_muestra = len(muestra)
        pct_umbral = {u: round(100 * conteo_umbral[u] / n_muestra, 1) if n_muestra else 0.0 for u in UMBRALES_COTA_PX}

        _diag_grupos.append({
            "n_segmentos": len(grupo),
            "span_m": round(span_px * mpx, 2),
            "bbox_ancho_m": round((max(xs) - min(xs)) * mpx, 2),
            "bbox_alto_m": round((max(ys) - min(ys)) * mpx, 2),
            "ancho_linea_min": round(min(anchos), 3),
            "ancho_linea_mediana": round(_diag_stats.median(anchos), 3),
            "ancho_linea_max": round(max(anchos), 3),
            "pct_cerca_cota_30px": pct_umbral[30],
            "pct_cerca_cota_60px": pct_umbral[60],
            "pct_cerca_cota_100px": pct_umbral[100],
            "n_muestra_cota": n_muestra,
        })
    _diag_grupos.sort(key=lambda g: -g["span_m"])
    print(f"  DIAGNOSTICO EJES/COTAS (temporal, no afecta el resultado): "
          f"{len(_diag_grupos)} grupos protegidos como muro, top 25 por span "
          f"({len(cotas_texto)} textos de cota disponibles para el chequeo de distancia):")
    for _g in _diag_grupos[:25]:
        print(f"     span={_g['span_m']}m  bbox={_g['bbox_ancho_m']}x{_g['bbox_alto_m']}m  "
              f"n_seg={_g['n_segmentos']}  ancho_linea(min/mediana/max)="
              f"{_g['ancho_linea_min']}/{_g['ancho_linea_mediana']}/{_g['ancho_linea_max']}  "
              f"%cerca_cota(30/60/100px, n={_g['n_muestra_cota']})="
              f"{_g['pct_cerca_cota_30px']}/{_g['pct_cerca_cota_60px']}/{_g['pct_cerca_cota_100px']}")


    # ── Paso 2.5 (NUEVO 2026-07-27): desproteger ACHURADO -- relleno de
    #    rampas/escaleras dibujado como muchos trazos cortos PARALELOS entre
    #    si (no colineales -- eso ya lo cubre el Paso 3 de guiones). Cada
    #    trazo individual de un achurado, o una cadena corta de 2-3 que por
    #    casualidad comparten un extremo, puede superar el span de 1.5m del
    #    Paso 2 sin ser parte de un muro real -- eso fragmentaba el interior
    #    de la rampa/escalera en muchas piezas chicas, dejando como
    #    'recinto_geo' solo el fragmento mas grande sobreviviente (visto en
    #    el run real del 2026-07-26: "Rampa Acceso Universal" midio 0.22m de
    #    ancho segun OpenCV cuando el plano indica 1.3m -- bbox de solo
    #    39x797px, una tira, no el area completa de la rampa).
    #    Señal distintiva de achurado vs. muro real: MUCHOS segmentos (5+)
    #    con angulo similar entre si, cercanos en el espacio (no necesitan
    #    tocarse por los extremos como un muro, un achurado se dibuja como
    #    trazos sueltos y paralelos). Un muro real rara vez tiene 5+ tramos
    #    paralelos sin ortogonales cerca (las esquinas rompen el patron).
    #
    #    RIESGO IDENTIFICADO Y MITIGADO: este mismo plano tiene achurado
    #    diagonal real sobre TRAMOS DE MURO para indicar "Se retira"
    #    (amarillo) / "Se construye" (rojo) -- ver simbologia del plano. Un
    #    achurado asi corre PEGADO a lo largo del muro (banda angosta, ancho
    #    ~= espesor de muro, 10-20cm) mientras que el achurado de RELLENO de
    #    una rampa/escalera cubre un AREA 2D completa (ambas dimensiones
    #    grandes, ancho de rampa ~1-1.5m). Se agrega un chequeo de
    #    "extension perpendicular" al angulo dominante del grupo: si los
    #    segmentos estan comprimidos en una banda angosta (<80px perpendicular)
    #    se asume achurado DE MURO (banda) y NO se desprotege -- solo se
    #    desprotege si el grupo cubre una extension perpendicular amplia
    #    (relleno 2D real). 80px equivale a ~47cm en el plano de prueba (MPX
    #    de este PDF, calculado desde 'largo_max_m'/bbox_h_px de la rampa) --
    #    el equivalente real en cm varia con la escala/DPI de cada plano, no
    #    es un valor fijo; queda con margen razonable sobre un espesor de
    #    muro tipico (10-20cm) para este caso, pero conviene revisar si se
    #    usa con planos a una escala muy distinta. Sin esto, el fix podria
    #    borrar muros reales marcados con achurado de intervencion y repetir
    #    la regresion catastrofica de fusion de recintos ya vista antes esta
    #    sesion.
    TOL_ACHURADO_ANGULO_DEG = 8
    RADIO_ACHURADO_PX = 60
    MIN_SEGMENTOS_ACHURADO = 5
    MIN_EXTENSION_PERPENDICULAR_PX = 80

    def _cerca_y_paralelo(s1, s2):
        dif_ang = abs(_angulo_segmento(s1) - _angulo_segmento(s2))
        dif_ang = min(dif_ang, 180 - dif_ang)
        if dif_ang > TOL_ACHURADO_ANGULO_DEG:
            return False
        m1 = ((s1['p1'][0] + s1['p2'][0]) / 2, (s1['p1'][1] + s1['p2'][1]) / 2)
        m2 = ((s2['p1'][0] + s2['p2'][0]) / 2, (s2['p1'][1] + s2['p2'][1]) / 2)
        return _distancia(m1, m2) <= RADIO_ACHURADO_PX

    def _angulo_promedio_180(angs):
        # FIX 2026-07-27 (encontrado en re-revision, antes de correr en Colab):
        # promediar angulos con suma directa (sum(angs)/len(angs)) esta MAL para
        # una magnitud 180-periodica como esta (orientacion de linea sin
        # direccion, ver _angulo_segmento). Si el grupo mezcla, por ejemplo,
        # ~2 grados y ~178 grados -- que son casi la MISMA orientacion, apenas
        # 4 grados de diferencia real -- la suma directa promedia a 90 grados,
        # perpendicular a ambas, un resultado completamente equivocado. Se usa
        # la media circular estandar para datos axiales: duplicar el angulo
        # (180-periodico -> 360-periodico), promediar como vector (seno/coseno),
        # volver a la mitad.
        suma_sin = sum(math.sin(math.radians(2 * a)) for a in angs)
        suma_cos = sum(math.cos(math.radians(2 * a)) for a in angs)
        return (math.degrees(math.atan2(suma_sin, suma_cos)) / 2) % 180

    def _extension_perpendicular(grupo):
        angs = [_angulo_segmento(segmentos_l[i]) for i in grupo]
        ang_prom = _angulo_promedio_180(angs)
        rad_perp = math.radians(ang_prom + 90)
        ux, uy = math.cos(rad_perp), math.sin(rad_perp)
        proys = []
        for i in grupo:
            s = segmentos_l[i]
            mx = (s['p1'][0] + s['p2'][0]) / 2
            my = (s['p1'][1] + s['p2'][1]) / 2
            proys.append(mx * ux + my * uy)
        return max(proys) - min(proys)

    # REVERTIDO 2026-07-27 -- confirmado en Colab que este fix causa una
    # regresion grave: en la corrida real (archicheck_geometrico_pdv_26jul_2232)
    # desprotegio 569 de 1521 segmentos (37%) en una pagina y 579 en la otra --
    # muy por encima de lo que cualquier achurado real de rampa/escalera podria
    # explicar. Volvio a aparecer la fusion catastrofica de ~140m2 que ya se
    # habia peleado antes esta sesion (el filtro de 'recintos_excluidos_por_
    # fusion' la atrapo esta vez y no corrompio el resultado final, pero el
    # hecho de que reaparezca confirma que se estan rompiendo muros reales en
    # algun lado, no solo achurado). Causa raiz identificada: la salvaguarda de
    # "extension perpendicular" (pensada para exigir que el achurado cubra un
    # area 2D LOCAL y acotada) no protege contra el encadenamiento TRANSITIVO
    # de Union-Find -- si hay una cadena de segmentos de angulo parecido
    # conectando puntos distantes entre si (muy probable en un plano
    # rectilineo, donde la mayoria de los muros estan a 0/90 grados), el grupo
    # completo puede terminar abarcando gran parte de la pagina, y un grupo asi,
    # disperso, tambien "aprueba" el chequeo de extension perpendicular amplia
    # -- no porque haya relleno 2D real en ningun punto local, sino porque el
    # grupo mismo es enorme y disperso. Es un error de diseño (el supuesto de
    # "cluster local acotado" no se sostiene con agrupamiento transitivo), no
    # un problema de calibrar mejor los umbrales -- se desactiva la
    # desproteccion en vez de intentar otro ajuste sin poder probarlo. Queda
    # ACTIVO el conteo diagnostico (n_achurado_desprotegido) para ver cuantos
    # segmentos habria tocado, sin tocarlos, por si sirve para un rediseño
    # futuro (ej. exigir que el cluster ademas sea compacto en su propio
    # bounding box, no solo que el ultimo par de vecinos este cerca).
    ACHURADO_DESPROTEGER_ACTIVO = False
    grupos_achurado = _agrupar_segmentos(segmentos_l, RADIO_ACHURADO_PX, _cerca_y_paralelo) if segmentos_l else []
    n_achurado_desprotegido = 0
    for grupo in grupos_achurado:
        if len(grupo) < MIN_SEGMENTOS_ACHURADO:
            continue
        if _extension_perpendicular(grupo) < MIN_EXTENSION_PERPENDICULAR_PX:
            continue  # banda angosta -- probable achurado de muro, no se toca
        if not ACHURADO_DESPROTEGER_ACTIVO:
            n_achurado_desprotegido += len([i for i in grupo if protegido[i]])
            continue
        for i in grupo:
            if protegido[i]:
                protegido[i] = False
                n_muro_protegido -= 1
                n_achurado_desprotegido += 1

    # ── Paso 3: entre los NO protegidos, agrupar por colinealidad + gap
    #    regular (patron de guiones dibujado a mano, ya que path['dashes']
    #    no sirve en este PDF — ver diagnostico 2026-07-24 en el roadmap).
    #    NOTA 2026-07-31: las constantes y _colineal_y_cerca se movieron
    #    antes del Paso 2 (ver "Paso 1.5" mas arriba) para poder detectar
    #    EJES antes de que el Paso 2 los proteja por conectividad como si
    #    fueran muro. Aqui se reutilizan sin cambios.
    idx_no_protegidos = [i for i in range(len(segmentos_l)) if not protegido[i]]
    segmentos_candidatos = [segmentos_l[i] for i in idx_no_protegidos]

    grupos_dash = _agrupar_segmentos(segmentos_candidatos, TOL_DASH_GAP_PX, _colineal_y_cerca) if segmentos_candidatos else []

    es_dash_local = [False] * len(segmentos_candidatos)
    muestras_cadenas = []
    for grupo in grupos_dash:
        if len(grupo) < MIN_SEGMENTOS_CADENA:
            continue
        span = _span_grupo(segmentos_candidatos, grupo)
        if span < UMBRAL_DASH_PX:
            continue
        largos = [_distancia(segmentos_candidatos[i]['p1'], segmentos_candidatos[i]['p2']) for i in grupo]
        promedio = sum(largos) / len(largos)
        variacion = (max(largos) - min(largos)) / promedio if promedio else 999
        if variacion > 0.9:  # muy irregular, no parece un patron de guiones real
            continue
        for i in grupo:
            es_dash_local[i] = True
        if len(muestras_cadenas) < 5:
            muestras_cadenas.append({'n_segmentos': len(grupo), 'span_m': round(span * mpx, 2)})

    # ── Paso 4: armar las 3 listas de salida ────────────────────
    trazos = []
    lineas_discontinuas = []

    for local_i, s in enumerate(segmentos_candidatos):
        p1a = ajustar(*s['p1']); p2a = ajustar(*s['p2'])
        item = {
            'puntos': [(round(p1a[0]), round(p1a[1])), (round(p2a[0]), round(p2a[1]))],
            'ancho_linea': round(s['ancho_linea'], 2),
        }
        if es_dash_local[local_i]:
            lineas_discontinuas.append(item)
        else:
            trazos.append({'tipo': 'l', **item})

    # AJUSTE 2026-07-25: 're'/'qu' (rectangulo/quad) nunca participan de la
    # cadena de conectividad de 'l' -- antes solo se protegian si su propio
    # bbox ya era grande, lo que deja a cualquier 'qu'/'re' chico sin
    # proteccion pase lo que pase, aunque sea parte de un muro real (solo
    # hay 41 'qu' en este PDF, ninguno 're' -- ver diagnostico 2026-07-24).
    # Se decide NO borrarlos nunca: son pocos, el riesgo de que alguno sea
    # muro real no vale el ahorro de limpieza de simbolos. Las curvas ('c')
    # si se siguen borrando siempre -- un muro nunca se dibuja como curva.
    for op, pts, ancho_linea in otros_items:
        if op != 'c':
            continue  # 're'/'qu' protegidos siempre, ver nota arriba
        pts_ajustados = [ajustar(x, y) for x, y in pts]
        trazos.append({
            'tipo': op,
            'puntos': [(round(x), round(y)) for x, y in pts_ajustados],
            'ancho_linea': round(ancho_linea, 2),
        })

    return {
        'cotas_texto': cotas_texto,
        'trazos': trazos,
        'lineas_discontinuas': lineas_discontinuas,
        'n_texto': len(cotas_texto),
        'n_trazos': len(trazos),
        'n_muro_protegido': n_muro_protegido,
        'n_achurado_desprotegido': n_achurado_desprotegido,
        'n_lineas_discontinuas': len(lineas_discontinuas),
        'n_cadenas_discontinuas': len(muestras_cadenas),
        'diagnostico_muestra_cadenas_discontinuas': muestras_cadenas,
        'muros_geo': muros_geo,
        'puertas_geo': puertas_geo,
    # DIAGNOSTICO (2026-08-04, SOLO diagnostico -- no agrega ninguna puerta
    # nueva ni cambia ningun resultado): el usuario encontro puertas reales
    # dibujadas en un color MUY tenue que no generan ningun trazo vectorial via
    # get_drawings() (0 trazos 'l' o 'c' encontrados en esas zonas, confirmado
    # buscando en el JSON). Hipotesis sin confirmar: podrian ser imagenes
    # incrustadas (un icono raster) en vez de geometria vectorial. Esto solo
    # imprime que imagenes hay y donde caen en pixeles (mismo sistema de
    # coordenadas que muros_geo/puertas_geo) para poder comparar a mano contra
    # las ubicaciones reales -- NO se agrega nada a muros_geo/puertas_geo a
    # partir de esto todavia. Si hay duda sobre si algo encontrado es una
    # puerta real, debe quedar para que el arquitecto lo confirme, no asumirse.
    try:
        _imgs_pagina = pdf_page.get_images(full=True)
        print(f'  DIAGNOSTICO IMAGENES: {len(_imgs_pagina)} imagen(es) incrustada(s) en el PDF de esta pagina (sin filtrar por crop)')
        for _img_info in _imgs_pagina:
            _xref = _img_info[0]
            _rects = pdf_page.get_image_rects(_xref)
            for _r in _rects:
                _p0px = to_px(fitz.Point(_r.x0, _r.y0))
                _p1px = to_px(fitz.Point(_r.x1, _r.y1))
                _cx = (_p0px[0] + _p1px[0]) / 2
                _cy = (_p0px[1] + _p1px[1]) / 2
                if not dentro_crop(_cx, _cy):
                    continue
                _x0, _y0 = ajustar(*_p0px)
                _x1, _y1 = ajustar(*_p1px)
                _ancho_m = round(abs(_x1 - _x0) * mpx, 2)
                _alto_m = round(abs(_y1 - _y0) * mpx, 2)
                print(f'    xref={_xref} bbox_px=({round(min(_x0,_x1))},{round(min(_y0,_y1))})-({round(max(_x0,_x1))},{round(max(_y0,_y1))}) tamano_m={_ancho_m}x{_alto_m}')
    except Exception as _e_img:
        print(f'  DIAGNOSTICO IMAGENES: error al inspeccionar - {_e_img}')
    }

# ══════════════════════════════════════════════════════════
# NUEVO 2026-07-27 — Deteccion automatica de figuras por lamina
#   Reduce el crop/escala manual: analiza la lamina COMPLETA (sin recortar)
#   y sugiere que poner en PAGINAS_Y_ESCALAS, en vez de que el usuario abra
#   el PDF y adivine fracciones de crop a ojo.
# ══════════════════════════════════════════════════════════
def detectar_figuras_lamina(imagen_rgb, numero_pagina=None, worker_url=WORKER_URL, max_dim=1600):
    """
    OPCIONAL, no se llama automaticamente desde el loop principal todavia.

    Por que existe: hoy el crop pasa ANTES que Claude Vision vea la imagen
    (ver 'plano_full' -> 'plano' mas abajo) -- Claude nunca mira la lamina
    completa, solo el recorte de planta que el usuario ya eligio a mano. Por
    eso 'escalas_detectadas' del prompt principal casi siempre trae una sola
    escala: no es que Claude haya confirmado que la lamina completa tiene una
    sola escala, es que solo le mostramos un pedazo que ya sabemos que la
    tiene.

    Que hace: manda la lamina COMPLETA (imagen_rgb, sin recortar) a Claude
    Vision y le pide identificar cada dibujo/figura presente -- planta,
    corte, elevacion, emplazamiento, cuadro, detalle -- con su tipo, bbox
    relativo (0-1) y escala asociada. Con eso arma e imprime una sugerencia
    de tuplas para pegar en PAGINAS_Y_ESCALAS (celda anterior).

    NO reemplaza el crop manual todavia, y NO se aplica solo -- requiere
    revision humana antes de usarse, mismo criterio que el resto del
    pipeline (nada de Claude Vision se usa sin validar). Casos como
    emplazamiento/corte/elevacion/cuadro NO necesitan escala de este tipo
    (no alimentan el motor de reglas geometrico) -- se listan igual, para
    que el usuario sepa que hay en la lamina, pero no generan sugerencia de
    PAGINAS_Y_ESCALAS (esa lista es solo para figuras tipo 'planta').

    Uso (antes de definir PAGINAS_Y_ESCALAS, con 'paginas' ya cargado):
        detectar_figuras_lamina(paginas[1], numero_pagina=2)   # pagina 2 (indice 1)
    'numero_pagina' es solo para que la sugerencia impresa quede lista para
    copiar y pegar (con el numero real en vez de un placeholder) -- si se
    omite, la sugerencia usa 'PAGINA_PLANTA' como texto a completar a mano.

    Devuelve la lista cruda de figuras detectadas (sin procesar), por si se
    quiere inspeccionar directamente en vez de leer los prints.
    """
    print('  → Detectando figuras en la lámina completa (sin recortar)...')
    h_full, w_full = imagen_rgb.shape[:2]
    factor_resize = min(1.0, max_dim / max(h_full, w_full))
    img_deteccion = (cv2.resize(imagen_rgb, (int(w_full * factor_resize), int(h_full * factor_resize)))
                      if factor_resize < 1.0 else imagen_rgb)

    _, buf = cv2.imencode('.png', cv2.cvtColor(img_deteccion, cv2.COLOR_RGB2BGR))
    img_b64 = base64.standard_b64encode(buf.tobytes()).decode()

    PROMPT_FIGURAS = (
        'Eres un asistente que indexa laminas de planos arquitectonicos chilenos. '
        'Esta imagen es una lamina COMPLETA de un expediente de planos -- normalmente '
        'contiene varios dibujos distintos en la misma hoja (ej. una o mas plantas de '
        'arquitectura de distintos niveles, un plano de emplazamiento, cortes, '
        'elevaciones, cuadros de superficie o normativos, detalles constructivos), cada '
        'uno posiblemente a una escala grafica distinta.\n'
        'Devuelve SOLO JSON puro sin markdown ni texto extra:\n'
        '{"figuras":[{"tipo":"planta|corte|elevacion|emplazamiento|cuadro|detalle|otro",'
        '"descripcion":"ej. Situacion Propuesta Nivel 1, Corte A-A, Cuadro de Superficies",'
        '"escala":"ej. 1:50, o null si no aplica (cuadros/detalles sin escala grafica propia)",'
        '"bbox_relativo":{"x1":0.0,"y1":0.0,"x2":1.0,"y2":1.0},'
        '"confianza":"alta|media|baja"}]}\n'
        'bbox_relativo: recuadro que encierra SOLO ese dibujo (no toda la lamina), como '
        'fraccion del ancho/alto total de la imagen (0.0=borde superior/izquierdo, '
        '1.0=borde inferior/derecho), con margen suficiente para no cortar cotas o textos '
        'que pertenezcan a ese dibujo. Si una lamina tiene mas de un nivel de planta '
        'dibujado por separado (ej. Nivel 1 y Nivel 2), son figuras DISTINTAS, cada una '
        'con su propio bbox_relativo -- no las mezcles en una sola. Si dos dibujos se '
        'superponen o es dificil separar donde termina uno y empieza otro, usa '
        'confianza "baja" en vez de inventar un limite.'
    )

    try:
        resp = requests.post(
            worker_url,
            json={'messages': [{'role': 'user', 'content': [
                {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': img_b64}},
                {'type': 'text', 'text': PROMPT_FIGURAS}
            ]}]},
            timeout=120, stream=True
        )
        resp.raise_for_status()
        raw_text = ''
        for line in resp.iter_lines():
            if not line:
                continue
            line = line.decode('utf-8') if isinstance(line, bytes) else line
            if not line.startswith('data: '):
                continue
            payload = line[6:].strip()
            if not payload or payload == '[DONE]':
                continue
            try:
                evt = json.loads(payload)
                if (evt.get('type') == 'content_block_delta' and
                        evt.get('delta', {}).get('type') == 'text_delta'):
                    raw_text += evt['delta']['text']
            except Exception:
                pass
        m = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if not m:
            print('  ⚠ Deteccion de figuras: sin JSON en la respuesta')
            return []
        figuras = json.loads(m.group()).get('figuras', [])
    except Exception as e:
        print(f'  ⚠ Error detectando figuras: {e}')
        return []

    print(f'  ✓ {len(figuras)} figura(s) detectada(s):')
    sugerencias_planta = []
    for f in figuras:
        tipo = f.get('tipo', '?')
        desc = f.get('descripcion', '')
        esc  = f.get('escala') or '—'
        conf = f.get('confianza', '?')
        bbox = f.get('bbox_relativo') or {}
        print(f'     [{tipo:<11}] {desc}  |  escala {esc}  |  confianza {conf}  |  '
              f'bbox ({bbox.get("x1")}, {bbox.get("y1")}, {bbox.get("x2")}, {bbox.get("y2")})')
        if tipo == 'planta' and esc != '—' and all(k in bbox for k in ('x1', 'y1', 'x2', 'y2')):
            sugerencias_planta.append((desc, esc, (bbox['x1'], bbox['y1'], bbox['x2'], bbox['y2']), conf))

    if sugerencias_planta:
        pagina_txt = str(numero_pagina) if numero_pagina is not None else 'PAGINA_PLANTA'
        print('\n  📋 Sugerencia para PAGINAS_Y_ESCALAS — REVISAR antes de usar, no se aplica solo:')
        for desc, esc, crop_sug, conf in sugerencias_planta:
            nota = '' if conf == 'alta' else f'  # confianza {conf}, revisar a ojo antes de confiar'
            print(f"     ({pagina_txt}, '{esc}', {crop_sug}),{nota}  # {desc}")
    else:
        print('  (sin figuras tipo "planta" con escala detectada — nada que sugerir para PAGINAS_Y_ESCALAS)')

    return figuras

resultados_paginas = []
viz_pages          = []

entries = [(e[0], e[1], e[2] if len(e) > 2 else None) for e in PAGINAS_Y_ESCALAS]

# ── Cuadro de superficies (2026-07-31) — extraccion de texto una sola vez,
#    fuera del loop de paginas, via PyMuPDF (mismo metodo confiable que ya
#    usa cotas_texto). No depende de que Claude Vision lea la imagen para
#    los numeros -- el texto real del cuadro se le entrega ya extraido.
TEXTO_CUADRO_SUPERFICIES = ''
if PAGINA_CUADRO_SUPERFICIES:
    if 1 <= PAGINA_CUADRO_SUPERFICIES <= len(doc):
        TEXTO_CUADRO_SUPERFICIES = doc[PAGINA_CUADRO_SUPERFICIES - 1].get_text().strip()
        if TEXTO_CUADRO_SUPERFICIES:
            print(f'  ✓ Cuadro de superficies: {len(TEXTO_CUADRO_SUPERFICIES)} caracteres extraidos de la pagina {PAGINA_CUADRO_SUPERFICIES}')
        else:
            print(f'  ⚠ Cuadro de superficies: la pagina {PAGINA_CUADRO_SUPERFICIES} no devolvio texto extraible (¿es una imagen escaneada?)')
    else:
        print(f'  ⚠ PAGINA_CUADRO_SUPERFICIES={PAGINA_CUADRO_SUPERFICIES} fuera de rango (PDF tiene {len(doc)} paginas) — se ignora')

# Precalcular cuántas veces aparece cada página (para nombres de archivo únicos)
page_count = {}
for pag, _, _ in entries:
    page_count[pag] = page_count.get(pag, 0) + 1
page_idx_so_far = {}

for (PAGINA_PLANTA, ESCALA_MANUAL, crop) in entries:
    print(f'\n{"="*56}')
    print(f'  Página {PAGINA_PLANTA}  —  escala {ESCALA_MANUAL}')
    if crop:
        print(f'  Recorte: ({crop[0]:.0%},{crop[1]:.0%}) → ({crop[2]:.0%},{crop[3]:.0%})')
    print(f'{"="*56}')

    if PAGINA_PLANTA < 1 or PAGINA_PLANTA > len(paginas):
        print(f'  ⚠ Página {PAGINA_PLANTA} fuera de rango (PDF tiene {len(paginas)} páginas). Saltando.')
        continue

    # Índice único por entrada (resuelve el caso de 2 crops de la misma página)
    entry_idx = len(resultados_paginas)
    page_idx_so_far[PAGINA_PLANTA] = page_idx_so_far.get(PAGINA_PLANTA, 0) + 1
    sub_idx = page_idx_so_far[PAGINA_PLANTA]
    fname_tag = (f'pag{PAGINA_PLANTA}-{sub_idx}'
                 if page_count[PAGINA_PLANTA] > 1
                 else f'pag{PAGINA_PLANTA}')

    plano_full = paginas[PAGINA_PLANTA - 1]
    h_f, w_f   = plano_full.shape[:2]

    # Aplicar recorte si está definido
    if crop:
        x1f, y1f, x2f, y2f = crop
        x1 = int(x1f * w_f); y1 = int(y1f * h_f)
        x2 = int(x2f * w_f); y2 = int(y2f * h_f)
        plano = plano_full[y1:y2, x1:x2].copy()
    else:
        plano = plano_full

    h, w  = plano.shape[:2]
    scale_ratio = int(ESCALA_MANUAL.split(':')[1])
    MPX   = 0.0254 * scale_ratio / DPI
    M2_PX = MPX ** 2
    print(f'  {w}x{h} px analizados  |  {MPX:.5f} m/px  |  1m = {int(1/MPX):,} px')

    # ── 1. Claude Vision ────────────────────────────────────
    print('  → Claude Vision...')
    _, buf_orig = cv2.imencode('.png', cv2.cvtColor(plano, cv2.COLOR_RGB2BGR))
    img_b64_orig = base64.standard_b64encode(buf_orig.tobytes()).decode()

    # Segunda imagen: version con contraste/nitidez mejorada, como referencia
    # adicional para que Claude vuelva a mirar puertas/ventanas dificiles de ver.
    plano_mejorado = mejorar_contraste_nitidez(plano)
    _, buf_mejor = cv2.imencode('.png', cv2.cvtColor(plano_mejorado, cv2.COLOR_RGB2BGR))
    img_b64_mejor = base64.standard_b64encode(buf_mejor.tobytes()).decode()

    # FIX 2026-07-26: campos agregados tras comparar contra el analisis real de
    # Revi sobre este mismo plano ("Revision planos Revi completo.txt", ver
    # roadmap seccion "Benchmark directo Revi vs ArchiCheck"). Revi demostro
    # leer: sentido de apertura de puertas, circulo de giro en banos accesibles,
    # el cuadro de superficies oficial de la lamina, y detectar cuando una
    # lamina trae mas de una escala. Estos 4 campos son SOLO de lectura/extraccion
    # (igual que Revi) — la diferenciacion real esta en que, mas abajo, cruzamos
    # 'cuadro_superficies_oficial' contra el area medida por OpenCV (geometria
    # independiente), y 'circulo_giro_m' contra una regla de accesibilidad, en
    # vez de solo transcribir lo que dice el plano.
    PROMPT = (
        f'Eres revisor DOM experto en OGUC, LGUC y DDU (Chile). '
        f'Analiza este plano arquitectonico a escala {ESCALA_MANUAL}.\n'
        'Te doy DOS imagenes del mismo plano: la primera es la imagen original a color, '
        'la segunda es una version con contraste y nitidez realzados (en blanco y negro) — '
        'usa la segunda especificamente para buscar puertas y ventanas que sean dificiles '
        'de distinguir en la primera por el relleno de color de los recintos.\n'
        'Devuelve SOLO JSON puro sin markdown ni texto extra:\n'
        '{"tipo_plano":"planta|corte|elevacion|detalle|otro",'
        '"uso_del_proyecto":"restaurante|vivienda|oficina|comercio|equipamiento|otro",'
        '"nivel":"descripcion o null",'
        '"escalas_detectadas":["1:50"],'
        '"recintos":[{"nombre":"...","tipo":"sala|cocina|bano|bodega|pasillo|terraza|bar|oficina|rampa|escalera|otro",'
        '"etiqueta_en_plano":"texto exacto o null","area_estimada_m2":null,"ancho_estimado_m":null,'
        '"cx_relativo":0.5,"cy_relativo":0.5,"cumple_oguc":true,"observacion":"o null",'
        '"es_accesible_universal":false,"circulo_giro_1_50_detectado":null}],'
        '"elementos_detectados":{"puertas":0,"ventanas":0,"escaleras":0,"rampas":0,"salidas_emergencia":0},'
        '"puertas_detalle":[{"id":"P01","ubicacion_o_recinto":"...","ancho_estimado_m":null,'
        '"sentido_apertura":"interior|exterior|no_determinado","cx_relativo":0.5,"cy_relativo":0.5,'
        '"p1_relativo":{"x":0.5,"y":0.5},"p2_relativo":{"x":0.5,"y":0.5}'
        '}],'
        '"ventanas_detalle":[{"id":"V01","ubicacion_o_recinto":"...","ancho_estimado_m":null,'
        '"cx_relativo":0.5,"cy_relativo":0.5,'
        '"p1_relativo":{"x":0.5,"y":0.5},"p2_relativo":{"x":0.5,"y":0.5}'
        '}],'
        '"escaleras_detalle":[{"id":"ES01","ubicacion_o_recinto":"...","ancho_estimado_m":null,'
        '"cx_relativo":0.5,"cy_relativo":0.5,'
        '"p1_relativo":{"x":0.5,"y":0.5},"p2_relativo":{"x":0.5,"y":0.5}'
        '}],'
        '"rampas_detalle":[{"id":"R01","ubicacion_o_recinto":"...","ancho_estimado_m":null,'
        '"cx_relativo":0.5,"cy_relativo":0.5,'
        '"p1_relativo":{"x":0.5,"y":0.5},"p2_relativo":{"x":0.5,"y":0.5}'
        '}],'
        '"cuadro_superficies_oficial":[{"recinto":"texto exacto del cuadro","area_m2_declarada":null}],'
        '"incumplimientos_oguc":[{"articulo":"","descripcion":"","gravedad":"ALTA|MEDIA|BAJA",'
        '"recinto_afectado":"","medida_requerida":"","medida_detectada":""}],'
        '"documentos_que_faltan":[],"resumen_ejecutivo":""}\n'
        'cx_relativo/cy_relativo: centroide del recinto como fraccion del ancho/alto '
        '(0.0=izquierda/arriba, 1.0=derecha/abajo). Aplica igual a cada item de '
        'puertas_detalle/ventanas_detalle/escaleras_detalle/rampas_detalle -- marca el '
        'centroide de CADA puerta/ventana/escalera/rampa individual que identifiques, '
        'no solo del recinto que la contiene. El campo "id" de cada item de estas 4 listas '
        'es un identificador corto propio (ej. P01, P02 para puertas; V01 para ventanas; '
        'ES01 para escaleras; R01 para rampas), unico dentro de esa lista en esta pagina. '
        'p1_relativo/p2_relativo: en puertas_detalle y ventanas_detalle, dos puntos que trazan '
        'el segmento real de la puerta/ventana en el plano (el ancho del vano/hoja tal como se '
        've dibujado en la lamina), no solo su centro -- si no puedes determinar el trazo exacto, '
        'usa tu mejor aproximacion visual del vano, nunca inventes una orientacion arbitraria. '
        'En escaleras_detalle y rampas_detalle, p1_relativo/p2_relativo son dos esquinas opuestas '
        'del rectangulo que delimita el elemento en el plano (no un trazo de linea). '
        'escalas_detectadas: TODAS las escalas graficas o numericas presentes en la lamina '
        '(una lamina puede traer varias escalas para distintos dibujos). '
        'cuadro_superficies_oficial: SOLO si el plano trae una tabla/cuadro impreso de '
        'superficies por recinto — dejar lista vacia si no existe tal cuadro (no inventar). '
        'circulo_giro_1_50_detectado: true/false solo si es_accesible_universal es true '
        '(bano/recinto rotulado como universal, accesible o para PMR); null en caso contrario.\n'
        'IMPORTANTE — texto que NUNCA es nombre de recinto: el texto de "rasante" (cotas de '
        'nivel de terreno/pendiente en cortes o emplazamiento, ej. "RASANTE +2.50", "NT +0.15") '
        'no representa un espacio habitable — no lo uses como "nombre" ni "etiqueta_en_plano" '
        'de ningun recinto, e ignoralo igual que ignorarias una cota de nivel suelta.'
        + (f'\n\nTEXTO EXTRAIDO DEL CUADRO DE SUPERFICIES IMPRESO (extraccion exacta via PDF, '
           f'no una lectura de imagen) — usa estos valores TAL CUAL para poblar '
           f'cuadro_superficies_oficial (recinto + area_m2_declarada), sin adivinar ni inventar '
           f'valores que no aparezcan en este texto. Si el recinto de esta planta no aparece '
           f'nombrado igual en el texto, no fuerces una coincidencia — deja el campo vacio para '
           f'ese recinto. Si este mismo texto ademas incluye un Estudio de Carga de Ocupacion '
           f'(personas por recinto/poligono, factor m2/persona), usa esos valores REALES para '
           f'calcular la carga de ocupacion en vez de estimarla — por ejemplo para determinar el '
           f'ancho minimo de escalera segun la tabla de OGUC Art. 4.2.10 (instruccion 3c mas '
           f'arriba), en vez de asumir el piso de la tabla por no poder calcular el aforo real:\n'
           f'{TEXTO_CUADRO_SUPERFICIES}\n' if TEXTO_CUADRO_SUPERFICIES else '')
    )

    analisis = {
        'tipo_plano': '?', 'uso_del_proyecto': '?', 'nivel': '?',
        'escalas_detectadas': [], 'recintos': [], 'elementos_detectados': {},
        'puertas_detalle': [], 'ventanas_detalle': [], 'escaleras_detalle': [], 'rampas_detalle': [],
        'cuadro_superficies_oficial': [],
        'incumplimientos_oguc': [], 'documentos_que_faltan': [],
        'resumen_ejecutivo': 'Sin analisis semantico'
    }
    try:
        resp = requests.post(
            WORKER_URL,
            json={'messages': [{'role': 'user', 'content': [
                {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': img_b64_orig}},
                {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': img_b64_mejor}},
                {'type': 'text', 'text': PROMPT}
            ]}]},
            timeout=180, stream=True
        )
        resp.raise_for_status()
        raw_text = ''
        for line in resp.iter_lines():
            if not line:
                continue
            line = line.decode('utf-8') if isinstance(line, bytes) else line
            if not line.startswith('data: '):
                continue
            payload = line[6:].strip()
            if not payload or payload == '[DONE]':
                continue
            try:
                evt = json.loads(payload)
                if (evt.get('type') == 'content_block_delta' and
                        evt.get('delta', {}).get('type') == 'text_delta'):
                    raw_text += evt['delta']['text']
            except:
                pass
        raw_text = raw_text.replace('```json', '').replace('```', '').strip()
        m = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if m:
            analisis  = json.loads(m.group())
            r_count   = len(analisis.get('recintos', []))
            inc_count = len(analisis.get('incumplimientos_oguc', []))
            print(f'  ✓ Claude: {r_count} recintos, {inc_count} incumplimientos')
        else:
            print('  ⚠ Claude: sin JSON en respuesta')
    except Exception as e:
        print(f'  ⚠ Error Claude: {e}')

    # ── 2. Extracción de datos vectoriales del PDF ──────────
    # NOTA 2026-07-23: se movio ANTES de OpenCV (antes iba despues) porque
    # OpenCV ahora necesita cotas_texto y lineas_discontinuas para limpiar
    # el raster antes de binarizar. Requiere que la Celda 2 haya confirmado
    # PDF vectorizado. `doc` sigue vivo en el kernel desde la Celda 2.
    print('  → Extracción vectorial...')
    pdf_page_actual = doc[PAGINA_PLANTA - 1]
    print(f'DIAGNOSTICO ROTACION: rotation={pdf_page_actual.rotation}  rect={pdf_page_actual.rect}  mediabox={pdf_page_actual.mediabox}')
    crop_px = (x1, y1, x2, y2) if crop else None
    datos_vectoriales = extraer_datos_vectoriales(pdf_page_actual, ZOOM, MPX, crop_px)
    print(f'  ✓ Vectorial: {datos_vectoriales["n_texto"]} textos (cotas/nombres), '
          f'{datos_vectoriales["n_trazos"]} trazos candidatos a símbolo, '
          f'{datos_vectoriales["n_muro_protegido"]} tramos protegidos como muro (por conectividad, no por largo), '
          f'{datos_vectoriales["n_achurado_desprotegido"]} tramos desprotegidos por parecer achurado (rampa/escalera), '
          f'{datos_vectoriales["n_lineas_discontinuas"]} en {datos_vectoriales["n_cadenas_discontinuas"]} cadena(s) de línea discontinua')
    if datos_vectoriales['diagnostico_muestra_cadenas_discontinuas']:
        print(f'     muestra de cadenas discontinuas detectadas: {datos_vectoriales["diagnostico_muestra_cadenas_discontinuas"]}')

    # ── 3. OpenCV — extracción geométrica ───────────────────
    print('  → OpenCV...')
    gray = cv2.cvtColor(plano, cv2.COLOR_RGB2GRAY)

    # FIX 2026-07-23 (a): borrar texto (cotas, nombres de recintos) usando
    # las posiciones exactas del vector — el texto NO debe limitar el area
    # de un recinto. Antes "0.8" o "Cocina" se trataban como si fueran parte
    # del muro porque adaptiveThreshold no distingue texto de linea.
    PADDING_TEXTO_PX = 3
    n_texto_borrado = 0
    for t in datos_vectoriales['cotas_texto']:
        tx0 = max(0, t['x'] - PADDING_TEXTO_PX)
        ty0 = max(0, t['y'] - PADDING_TEXTO_PX)
        tx1 = min(w, t['x'] + t['w'] + PADDING_TEXTO_PX)
        ty1 = min(h, t['y'] + t['h'] + PADDING_TEXTO_PX)
        if tx1 > tx0 and ty1 > ty0:
            gray[ty0:ty1, tx0:tx1] = 255
            n_texto_borrado += 1

    # FIX 2026-07-24 (b v2): borrar líneas discontinuas (deslinde, línea de
    # edificación, ejes). El intento original (v1, 2026-07-23) buscaba
    # path['dashes'] pero el diagnóstico confirmó que este PDF siempre lo
    # da sólido ('[] 0') — el punteado se dibuja a mano con muchos tramos
    # cortos separados, no con el atributo nativo de PDF. Ahora se detectan
    # por geometría (colinealidad + gap regular, ver extraer_datos_vectoriales).
    n_lineas_borradas = 0
    for ld in datos_vectoriales['lineas_discontinuas']:
        grosor_borrado = max(6, int(ld['ancho_linea'] * ZOOM) + 6)  # margen anti-aliasing
        pts = ld['puntos']
        for i in range(len(pts) - 1):
            cv2.line(gray, pts[i], pts[i + 1], 255, thickness=grosor_borrado)
        n_lineas_borradas += 1

    # FIX 2026-07-24 (c v2): borrar trazos cortos (artefactos, arcos de
    # puerta) del raster. El intento original (v1, 2026-07-23) filtraba por
    # LARGO individual (<3m) y causó una regresión grave: tramos reales de
    # muro perimetral en esquinas/quiebres también miden <3m y se borraban,
    # fusionando exterior+interior en un recinto falso de ~140-155 m² (ver
    # roadmap P1, capturas 2026-07-24). Ahora 'trazos' ya viene filtrado por
    # extraer_datos_vectoriales usando CONECTIVIDAD: un segmento corto que
    # está conectado a una cadena larga de muro real queda protegido y no
    # llega a esta lista — lo que sí llega es seguro de borrar.
    n_trazos_borrados = 0
    for tr in datos_vectoriales['trazos']:
        grosor_borrado = max(6, int(tr['ancho_linea'] * ZOOM) + 6)
        pts = tr['puntos']
        for i in range(len(pts) - 1):
            cv2.line(gray, pts[i], pts[i + 1], 255, thickness=grosor_borrado)
        if len(pts) >= 3:
            cv2.line(gray, pts[-1], pts[0], 255, thickness=grosor_borrado)
        n_trazos_borrados += 1

    print(f'  ✓ Limpieza pre-umbral: {n_texto_borrado} textos, {n_lineas_borradas} líneas discontinuas, '
          f'{n_trazos_borrados} trazos cortos (artefactos/arcos, ya excluye tramos de muro protegidos) borrados')

    binary_inv = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, blockSize=21, C=4)
    k_close    = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 25))
    muros      = cv2.dilate(binary_inv, k_close, iterations=2)
    k_open     = cv2.getStructuringElement(cv2.MORPH_RECT, (10, 10))
    limpios    = cv2.morphologyEx(cv2.bitwise_not(muros), cv2.MORPH_OPEN, k_open)

    n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        limpios, connectivity=8)

    # AJUSTE 2026-07-26: el "recinto" de ~65-141 m2 que veniamos persiguiendo
    # NO era causado por ninguno de los fixes de borrado vectorial -- ya
    # existia en la corrida ORIGINAL sin ningun cambio (65.92 m2 "Baño
    # Universal" en N1, 91.6 m2 "Pasillo" en N2, ver roadmap P1). Causa real:
    # una apertura real del plano (ej. la puerta de acceso principal) conecta
    # topologicamente el area de contexto/terreno (fuera del edificio, dentro
    # del deslinde) con el interior -- ahi no hay ningun trazo que borrar,
    # es un hueco real en el dibujo (una puerta ES una abertura). Ninguna
    # limpieza de texto/simbolos/lineas discontinuas puede arreglar esto,
    # porque el problema no es pixel de mas borrado, es la naturaleza de la
    # segmentacion por conectividad de OpenCV (una puerta abierta conecta
    # dos espacios "distintos" para nosotros pero son un solo blob de pixeles
    # para el algoritmo).
    # Fix: excluir cualquier componente que TOQUE el borde de la imagen Y
    # tenga un area implausible para un recinto individual real (>60 m2 --
    # el recinto real mas grande en el dataset de prueba es ~50 m2). Tocar
    # el borde solo no alcanza como criterio (Bodega=22m2 y Terraza=9m2 son
    # recintos reales y tambien tocan el borde en este recorte), por eso se
    # combinan ambas condiciones.
    UMBRAL_AREA_SOSPECHOSA_M2 = 60
    MARGEN_BORDE_PX = 2

    MIN_PX2 = int(0.5 / M2_PX)
    recintos_geo = []
    recintos_excluidos_por_fusion = []
    for idx in range(1, n_labels):
        area_px = int(stats[idx, cv2.CC_STAT_AREA])
        if area_px < MIN_PX2:
            continue
        area_m2 = round(area_px * M2_PX, 2)
        cx_abs  = int(centroids[idx][0])
        cy_abs  = int(centroids[idx][1])
        mask    = (labels == idx).astype(np.uint8) * 255
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        ancho_m = largo_m = None
        bbox = None
        if cnts:
            cnt  = max(cnts, key=cv2.contourArea)
            rect = cv2.minAreaRect(cnt)
            dims = sorted(rect[1])
            ancho_m = round(dims[0] * MPX, 2)
            largo_m = round(dims[1] * MPX, 2)
            bx, by, bw, bh = cv2.boundingRect(cnt)
            bbox = {'x': int(bx), 'y': int(by), 'w': int(bw), 'h': int(bh)}

        toca_borde = bbox is not None and (
            bbox['x'] <= MARGEN_BORDE_PX or bbox['y'] <= MARGEN_BORDE_PX or
            bbox['x'] + bbox['w'] >= w - MARGEN_BORDE_PX or
            bbox['y'] + bbox['h'] >= h - MARGEN_BORDE_PX
        )
        if toca_borde and area_m2 > UMBRAL_AREA_SOSPECHOSA_M2:
            recintos_excluidos_por_fusion.append({'area_m2': area_m2, 'bbox': bbox})
            continue

        recintos_geo.append({
            'id'         : f'E{len(recintos_geo)+1:02d}',
            'label'      : idx,
            'area_px'    : area_px,
            'area_m2'    : area_m2,
            'ancho_min_m': ancho_m,
            'largo_max_m': largo_m,
            'cx'         : cx_abs,
            'cy'         : cy_abs,
            'cx_rel'     : round(cx_abs / w, 3),
            'cy_rel'     : round(cy_abs / h, 3),
            'bbox'       : bbox,
        })
    recintos_geo.sort(key=lambda r: r['area_m2'], reverse=True)
    print(f'  ✓ OpenCV: {len(recintos_geo)} espacios >= 0.5 m²')
    if recintos_excluidos_por_fusion:
        areas_excluidas = ', '.join(f"{r['area_m2']}m²" for r in recintos_excluidos_por_fusion)
        print(f'  ⚠ {len(recintos_excluidos_por_fusion)} región(es) excluida(s) por parecer fusión exterior+interior '
              f'(toca el borde + área > {UMBRAL_AREA_SOSPECHOSA_M2}m²): {areas_excluidas}')

    # ── 4. Cruce semántica + geometría ──────────────────────
    recintos_claude = analisis.get('recintos', [])
    usados = set()

    def dist_rel(rg, rc):
        return math.sqrt((rg['cx_rel'] - rc.get('cx_relativo', 0.5))**2 +
                         (rg['cy_rel'] - rc.get('cy_relativo', 0.5))**2)

    def mejor_match(rg):
        cands = [(dist_rel(rg, rc), j, rc)
                 for j, rc in enumerate(recintos_claude) if j not in usados]
        if not cands:
            return None
        cands.sort(key=lambda x: x[0])
        d, j, rc = cands[0]
        if d < 0.25:
            usados.add(j)
            return rc
        return None

    tabla = []
    incumplimientos_geo = []

    for rg in recintos_geo:
        rc     = mejor_match(rg)
        # FIX 2026-07-23 (c): un recinto sin match de Claude Vision ya NO se
        # nombra en silencio ("Espacio E##") — se marca explicitamente para
        # que el arquitecto lo confirme (via la interfaz de validacion grafica
        # cuando exista; por ahora, print de advertencia + campo dedicado).
        sin_nombre = rc is None
        nombre = rc['nombre'] if rc else f'Espacio {rg["id"]} (SIN NOMBRE - confirmar con arquitecto)'
        tipo   = (rc['tipo'] if rc else 'otro').lower().split('/')[0].strip()
        area   = rg['area_m2']
        ancho  = rg['ancho_min_m']

        area_min, ancho_min, ref = OGUC_REGLAS.get(tipo, (None, None, None))
        area_ok = ancho_ok = None

        if area_min and area < area_min:
            area_ok = False
            incumplimientos_geo.append({
                'tipo': 'area', 'pagina': PAGINA_PLANTA,
                'recinto': nombre, 'id': rg['id'],
                'medido': area, 'minimo': area_min,
                'deficit': round(area_min - area, 2), 'ref': ref
            })
        if ancho_min and ancho is not None and ancho < ancho_min:
            ancho_ok = False
            incumplimientos_geo.append({
                'tipo': 'ancho', 'pagina': PAGINA_PLANTA,
                'recinto': nombre, 'id': rg['id'],
                'medido': ancho, 'minimo': ancho_min,
                'deficit': round(ancho_min - ancho, 2), 'ref': ref
            })

        # FIX 2026-07-26 (b, luego corregido en (c) mas abajo): regla real de
        # pendiente de rampa — Art. 4.1.7 N°2 OGUC. La version (b) de este fix se
        # baso en oguc_articulos.json (fuente curada, resumen incompleto) y asumio
        # una regla BINARIA (<=3m -> 12%, >3m -> 8%) -- ESO ESTABA MAL, corregido en
        # (c): el texto real (verificado contra oguc_pdf.json, extraccion completa
        # del PDF oficial) es una formula lineal continua, ver el comentario dentro
        # del bloque "if tipo == 'rampa':" para el detalle y la correccion.
        # 'largo_max_m' del recinto geometrico se usa como proxy del "desarrollo"
        # de la rampa (dimension mayor del rectangulo minimo que la contiene).
        if tipo == 'rampa':
            # FIX 2026-07-26 (revision exhaustiva post-implementacion): el patron
            # original '\d{1,2}[.,]\d{1,3}\s*%' exigia decimales -- una rampa
            # rotulada como "Pendiente 8%" (entero, sin coma) no matcheaba nunca y
            # el chequeo quedaba en silencio sin avisar. Ahora el decimal es
            # opcional. Ademas, para reducir falsos positivos (cualquier % cercano
            # a la rampa se tomaba como pendiente, aunque fuera de otra cosa), se
            # prioriza el texto que efectivamente dice "pendient..." junto al %;
            # solo si no aparece ese texto se usa un % suelto como respaldo.
            patron_pendiente = re.compile(r'(\d{1,2}(?:[.,]\d{1,3})?)\s*%')
            bbox_rg = rg.get('bbox')
            if bbox_rg:
                margen_px = int(0.6 / MPX) if MPX else 40
                bx0, by0 = bbox_rg['x'] - margen_px, bbox_rg['y'] - margen_px
                bx1, by1 = bbox_rg['x'] + bbox_rg['w'] + margen_px, bbox_rg['y'] + bbox_rg['h'] + margen_px
                pendientes_con_etiqueta = []
                pendientes_sueltas = []
                for t in datos_vectoriales['cotas_texto']:
                    m = patron_pendiente.search(t['texto'])
                    if not m:
                        continue
                    if bx0 <= t['x'] <= bx1 and by0 <= t['y'] <= by1:
                        valor = float(m.group(1).replace(',', '.'))
                        if 'pendient' in t['texto'].lower():
                            pendientes_con_etiqueta.append(valor)
                        else:
                            pendientes_sueltas.append(valor)
                pendientes_detectadas = pendientes_con_etiqueta or pendientes_sueltas
                if pendientes_detectadas:
                    pendiente_declarada = max(pendientes_detectadas)
                    desarrollo = rg.get('largo_max_m')
                    # FIX 2026-07-26 (c) -- CORRECCION IMPORTANTE: la regla binaria
                    # (<=3m->12%, >3m->8%) que se implemento antes estaba MAL. Se
                    # baso en el resumen de oguc_articulos.json (fuente curada,
                    # incompleta) que parafraseaba el articulo real de forma
                    # incorrecta. Al auditar contra oguc_pdf.json (770 articulos,
                    # extraccion completa del PDF oficial) aparecio el texto real y
                    # COMPLETO del Art. 4.1.7 N°2: "La pendiente de la rampa sera de
                    # un 8%, pudiendo llegar con esta a 9 m de largo. Para un largo
                    # de 1,5 m, la pendiente ira aumentando hasta alcanzar un 12%,
                    # como maximo... Para verificar la pendiente proyectada se usara
                    # la siguiente formula: i% = 12,8 - 0,5333*L" (L en metros,
                    # formula valida entre 1,5 y 9 m -- es la interpolacion lineal
                    # exacta entre los puntos (1,5m, 12%) y (9m, 8%) mencionados en
                    # el mismo parrafo, no un valor inventado). Es decir: la
                    # "formula progresiva" que cito el chatbot de Revi (y que esta
                    # sesion habia calificado de "fabricada") SI es real -- lo que
                    # estaba fabricado, en todo caso, eran los NUMEROS especificos
                    # que Revi calculo con ella (9,5%/9,60% en vez de los ~10,53%/
                    # ~10,40% que da la formula real para 4,25/4,50 m). Se corrige
                    # aqui a la formula real.
                    if desarrollo is None:
                        max_pendiente = 8.0
                    elif desarrollo <= 1.5:
                        max_pendiente = 12.0
                    elif desarrollo >= 9.0:
                        max_pendiente = 8.0
                    else:
                        max_pendiente = round(12.8 - 0.5333 * desarrollo, 2)
                    if pendiente_declarada > max_pendiente:
                        incumplimientos_geo.append({
                            'tipo': 'pendiente_rampa', 'pagina': PAGINA_PLANTA,
                            'recinto': nombre, 'id': rg['id'],
                            'medido': pendiente_declarada, 'minimo': None, 'maximo': max_pendiente,
                            'desarrollo_m': desarrollo,
                            'deficit': round(pendiente_declarada - max_pendiente, 2),
                            'ref': 'Art. 4.1.7 N°2 OGUC — pendiente max. 8% (desarrollo >=9m) a 12% (desarrollo <=1,5m), formula i%=12,8-0,5333*L entre esos valores'
                        })

        # FIX 2026-07-26: bano/recinto accesible sin circulo de giro -- Revi
        # demostro leer el circulo de giro en su analisis, pero solo lo
        # transcribe. Aqui ademas se convierte en una regla real (DDU 351 /
        # Art. 4.1.7 OGUC exige 1,50 m de diametro libre en recintos accesibles).
        if rc and rc.get('es_accesible_universal') and rc.get('circulo_giro_1_50_detectado') is False:
            incumplimientos_geo.append({
                'tipo': 'circulo_giro', 'pagina': PAGINA_PLANTA,
                'recinto': nombre, 'id': rg['id'],
                'medido': None, 'minimo': 1.50, 'deficit': None,
                'ref': 'DDU 351 / Art. 4.1.7 OGUC — circulo de giro 1,50 m en recintos accesibles'
            })

        # FIX 2026-07-26: cruce del cuadro de superficies oficial del plano (si
        # existe) contra el area MEDIDA por geometria independiente (OpenCV).
        # Esta validacion es la diferencia real frente a Revi: Revi transcribe
        # el cuadro de superficies pero no lo contrasta contra una medicion
        # propia del dibujo — aqui si, porque OpenCV mide el recinto sin
        # depender de lo que declare el cuadro.
        if rc:
            for fila_cuadro in analisis.get('cuadro_superficies_oficial', []):
                nombre_cuadro = (fila_cuadro.get('recinto') or '').strip().lower()
                if not nombre_cuadro:
                    continue
                nombre_rc = (rc.get('nombre') or '').strip().lower()
                if nombre_cuadro == nombre_rc or nombre_cuadro in nombre_rc or nombre_rc in nombre_cuadro:
                    area_declarada = fila_cuadro.get('area_m2_declarada')
                    if area_declarada and area > 0:
                        diff_pct = abs(area_declarada - area) / area * 100
                        if diff_pct > 15:
                            incumplimientos_geo.append({
                                'tipo': 'discrepancia_area_declarada', 'pagina': PAGINA_PLANTA,
                                'recinto': nombre, 'id': rg['id'],
                                'medido': area, 'declarado': area_declarada,
                                'diff_pct': round(diff_pct, 1),
                                'ref': 'Cuadro de superficies del plano vs. area medida por geometria (OpenCV)'
                            })
                    break

        tabla.append({
            'id'                   : rg['id'],
            'nombre'               : nombre,
            'tipo'                 : tipo,
            'pagina'               : PAGINA_PLANTA,
            'area_m2'              : area,
            'ancho_min_m'          : ancho,
            'largo_max_m'          : rg.get('largo_max_m'),
            'cumple_geo'           : (area_ok is not False) and (ancho_ok is not False),
            'cx_rel'               : rg['cx_rel'],
            'cy_rel'               : rg['cy_rel'],
            'bbox'                 : rg.get('bbox'),
            'sin_nombre_confirmar' : sin_nombre,
        })

    total   = round(sum(f['area_m2'] for f in tabla), 1)
    matched = sum(1 for f in tabla if not f['sin_nombre_confirmar'])
    print(f'  ✓ Cruce: {matched}/{len(tabla)} con nombre | {len(incumplimientos_geo)} incumpl. geo')

    sin_nombre_ids = [f['id'] for f in tabla if f['sin_nombre_confirmar']]
    if sin_nombre_ids:
        print(f'  ⚠ {len(sin_nombre_ids)} espacio(s) SIN NOMBRE — requieren que el arquitecto confirme qué son: {", ".join(sin_nombre_ids)}')

    resultados_paginas.append({
        'entry_idx'             : entry_idx,
        'fname_tag'             : fname_tag,
        'pagina'                : PAGINA_PLANTA,
        'escala'                : ESCALA_MANUAL,
        'crop'                  : list(crop) if crop else None,
        'analisis_semantico'    : analisis,
        'mediciones_geometricas': tabla,
        'muros_geo'             : datos_vectoriales.get('muros_geo', []),
        'puertas_geo'           : datos_vectoriales.get('puertas_geo', []),
        'incumplimientos_geo'   : incumplimientos_geo,
        'datos_vectoriales'     : datos_vectoriales,
        'recintos_excluidos_por_fusion': recintos_excluidos_por_fusion,
        'total_area_m2'         : total,
        'imagen_w_px'           : w,
        'imagen_h_px'           : h,
        'mpp'                   : MPX,
    })
    viz_pages.append({
        'entry_idx'   : entry_idx,
        'fname_tag'   : fname_tag,
        'pagina'      : PAGINA_PLANTA,
        'escala'      : ESCALA_MANUAL,
        'plano'       : plano,
        'labels'      : labels,
        'recintos_geo': recintos_geo,
        'w': w, 'h': h,
    })

print(f'\n{"="*56}')
total_inc = sum(len(p['incumplimientos_geo']) for p in resultados_paginas)
print(f'✓ Procesadas {len(resultados_paginas)} / {len(PAGINAS_Y_ESCALAS)} páginas')
print(f'  Incumplimientos geométricos totales: {total_inc}')


In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 4b — Cargar Grounding DINO + SAM 2  [GPU requerida]
#
# Ejecutar UNA VEZ por sesión de Colab.
# No es necesario re-ejecutar si vuelves a correr Celda 4.
# ══════════════════════════════════════════════════════════

# 1. Verificar GPU — obligatorio
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "\n⛔  GPU no disponible.\n"
        "    Ve a: Runtime → Change runtime type → Hardware accelerator: T4 GPU\n"
        "    Luego reinicia la sesión y ejecuta desde Celda 1.\n"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✓ GPU: {gpu_name}  ({vram_gb:.1f} GB VRAM)")

# 2. Instalar librerías
print("\nInstalando Grounding DINO + SAM 2 (~2–3 min la primera vez)...")
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers>=4.45.0",
     "git+https://github.com/facebookresearch/sam2.git"],
    check=True
)
print("✓ Librerías instaladas")

# 3. Cargar Grounding DINO (HuggingFace)
# size override: DINO por defecto procesa ~800px — subimos a 2048px para
# detectar elementos pequeños (puertas, ventanas) en planos de alta resolución
print("\nCargando Grounding DINO base (~700 MB) con resolución 2048px...")
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

DINO_ID        = "IDEA-Research/grounding-dino-base"
dino_processor = AutoProcessor.from_pretrained(
    DINO_ID,
    size={"shortest_edge": 2048, "longest_edge": 2048}
)
dino_model     = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_ID).to("cuda")
dino_model.eval()
print("✓ Grounding DINO listo (resolución máx. 2048px)")

# 4. Cargar SAM 2 — hiera-small (~185 MB)
print("\nCargando SAM 2 hiera-small (~185 MB)...")
from sam2.build_sam import build_sam2_hf
from sam2.sam2_image_predictor import SAM2ImagePredictor

sam2_predictor = SAM2ImagePredictor(build_sam2_hf("facebook/sam2.1-hiera-small"))
print("✓ SAM 2 listo")

vram_usada = torch.cuda.memory_allocated() / 1e9
print(f"\nVRAM usada: {vram_usada:.1f} / {vram_gb:.1f} GB  ({vram_usada/vram_gb*100:.0f}%)")
print("\n✓ Modelos listos. Ejecuta la Celda 4c.")

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 4c — Detección de elementos con DINO + SAM 2
#
# Detecta puertas, ventanas, escaleras, rampas, columnas
# en cada página ya procesada por OpenCV (Celda 4).
# Complementa la detección de recintos — no la reemplaza.
#
# Nota: DINO fue entrenado en fotografías. En planos 2D
# detecta símbolos reconocibles (puertas, ventanas) con
# ~60–75% de recall. Las mediciones son orientativas —
# confirmar siempre con cotas del plano.
# ══════════════════════════════════════════════════════════
from PIL import Image
import numpy as np, torch, cv2

# Prompt en inglés — DINO es más preciso en inglés para símbolos arquitectónicos
# FIX 2026-07-20: Grounding DINO agrupa frases por punto ".", y la convención
# oficial del modelo (README / model card IDEA-Research) es que el prompt
# TERMINE en punto. Sin el punto final, la última frase ("emergency exit")
# puede no agruparse bien en el post-proceso. Antes de este fix: sin punto final.
PROMPT         = "door . window . staircase . ramp . column . emergency exit ."
BOX_THRESHOLD  = 0.20   # ajustar: bajar a 0.15 si detecta poco; subir a 0.30 si hay falsos positivos
TEXT_THRESHOLD = 0.15

# DIAGNÓSTICO 2026-07-20: en las 5 corridas guardadas hasta ahora, DINO detectó
# en total 1 elemento (13% de confianza) en las 5 combinadas — muy por debajo
# del ~60-75% de recall documentado arriba. El fix del punto final puede ayudar,
# pero antes de asumir que fue la única causa, correr al menos una vez con
# DEBUG_THRESHOLD_MINIMO=True para ver si aparece CUALQUIER señal por debajo
# del threshold normal. Si con esto tampoco aparece nada, el problema probable-
# mente no es de threshold/prompt sino de brecha de dominio (DINO fue entrenado
# en fotografías, no en símbolos de líneas de plano CAD) — en ese caso lo más
# útil de este experimento es documentarlo y apoyarse más en el conteo semántico
# de Claude Vision (ya existe como fallback automático más abajo).
DEBUG_THRESHOLD_MINIMO = False
if DEBUG_THRESHOLD_MINIMO:
    BOX_THRESHOLD  = 0.05
    TEXT_THRESHOLD = 0.05
    print("⚠ MODO DIAGNÓSTICO: threshold bajado a 0.05 — esperar más falsos positivos, es solo para ver si hay señal real")

TIPO_ES = {
    "door"          : "puerta",
    "window"        : "ventana",
    "staircase"     : "escalera",
    "ramp"          : "rampa",
    "column"        : "columna",
    "emergency exit": "salida_emergencia",
}

# Dimensión máxima razonable por tipo (m) — superar esto = falso positivo
MAX_ANCHO_M = {
    "puerta"           : 2.5,   # ninguna puerta supera 2.5 m de hoja
    "ventana"          : 4.0,   # ventana corrida máx ~4 m
    "escalera"         : 6.0,   # escalera máx ~6 m de ancho
    "rampa"            : 4.0,   # rampa máx ~4 m de ancho
    "columna"          : 1.5,   # columna máx ~1.5 m
    "salida_emergencia": 3.0,
}
MIN_CONFIANZA = 0.15  # descartar detecciones con score < 15%

# Colores por tipo para visualización
COLORES_DINO = {
    "puerta"           : (30,  144, 255),
    "ventana"          : (0,   206, 209),
    "escalera"         : (255, 140,   0),
    "rampa"            : (148,   0, 211),
    "columna"          : (220,  20,  60),
    "salida_emergencia": (34,  139,  34),
}

print(f"Procesando {len(viz_pages)} página(s) con Grounding DINO + SAM 2...\n")

for vz in viz_pages:
    pag   = vz['pagina']
    plano = vz['plano']
    h, w  = plano.shape[:2]

    # Usar entry_idx para match directo — evita el bug cuando hay 2 crops de la misma página
    res_pag = resultados_paginas[vz['entry_idx']]
    MPX = res_pag['mpp']

    print(f"  Página {pag} [{vz['fname_tag']}]  |  {res_pag['escala']}  |  {w}×{h} px")

    # ── Grounding DINO ─────────────────────────────────────
    pil_img = Image.fromarray(plano)
    inputs  = dino_processor(
        images=pil_img, text=PROMPT, return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = dino_model(**inputs)

    detecciones = dino_processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold      = BOX_THRESHOLD,
        text_threshold = TEXT_THRESHOLD,
        target_sizes   = [(h, w)],
    )[0]

    boxes  = detecciones["boxes"].cpu().numpy()
    labels = detecciones["labels"]
    scores = detecciones["scores"].cpu().numpy()

    # Auto-retry con threshold reducido si no hay detecciones
    if len(boxes) == 0:
        retry_thr      = max(BOX_THRESHOLD * 0.65, 0.12)
        retry_text_thr = max(TEXT_THRESHOLD * 0.65, 0.10)
        print(f"    ⚠ Sin detecciones (thr={BOX_THRESHOLD:.2f}) — reintentando con thr={retry_thr:.2f}...")
        retry_det = dino_processor.post_process_grounded_object_detection(
            outputs, inputs.input_ids,
            threshold      = retry_thr,
            text_threshold = retry_text_thr,
            target_sizes   = [(h, w)],
        )[0]
        boxes  = retry_det["boxes"].cpu().numpy()
        labels = retry_det["labels"]
        scores = retry_det["scores"].cpu().numpy()
        if len(boxes) == 0:
            print(f"    ⚠ Sin detecciones incluso con threshold reducido — usando conteo semántico como respaldo\n")
            res_pag["elementos_dino"] = []
            vz["elementos_dino"]      = []
            continue
        print(f"    → {len(boxes)} candidato(s) con threshold reducido — aplicando filtro de calidad")

    # ── SAM 2 — máscara precisa por cada detección ─────────
    elementos = []
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
        sam2_predictor.set_image(plano)
        for box, label, score in zip(boxes, labels, scores):
            x1, y1, x2, y2 = float(box[0]), float(box[1]), float(box[2]), float(box[3])
            ancho_m = round((x2 - x1) * MPX, 3)
            alto_m  = round((y2 - y1) * MPX, 3)

            masks, _, _ = sam2_predictor.predict(
                box=np.array([x1, y1, x2, y2])[None],
                multimask_output=False,
            )
            area_mascara = round(float(masks[0].sum()) * MPX ** 2, 3)

            tipo_es = TIPO_ES.get(label, label)
            elementos.append({
                "tipo"           : tipo_es,
                "tipo_en"        : label,
                "confianza"      : round(float(score), 3),
                "bbox_px"        : [round(x1), round(y1), round(x2), round(y2)],
                "ancho_m"        : ancho_m,
                "alto_m"         : alto_m,
                "area_mascara_m2": area_mascara,
                "cx_rel"         : round((x1 + x2) / 2 / w, 3),
                "cy_rel"         : round((y1 + y2) / 2 / h, 3),
            })

    # ── Filtro de calidad — eliminar falsos positivos ───────
    n_bruto = len(elementos)
    elementos = [
        e for e in elementos
        if e["confianza"] >= MIN_CONFIANZA
        and e["ancho_m"] <= MAX_ANCHO_M.get(e["tipo"], 10.0)
    ]
    n_filtrados = n_bruto - len(elementos)
    if n_filtrados > 0:
        print(f"    ⚠ {n_filtrados} detección(es) descartada(s) (confianza < {MIN_CONFIANZA} o dimensión fuera de rango)")

    if not elementos:
        print(f"    ⚠ Sin detecciones válidas tras filtro — usando conteo semántico como respaldo\n")
        res_pag["elementos_dino"] = []
        vz["elementos_dino"]      = []
        continue

    res_pag["elementos_dino"] = elementos
    vz["elementos_dino"]      = elementos

    # Resumen
    conteo = {}
    for e in elementos:
        conteo[e["tipo"]] = conteo.get(e["tipo"], 0) + 1
    print(f"    ✓ {len(elementos)} detectado(s): {conteo}")

    # Alerta puertas angostas (OGUC Art. 4.2.2 — mín 0.90 m)
    angostas = [e for e in elementos if e["tipo"] == "puerta" and e["ancho_m"] < 0.90]
    if angostas:
        print(f"    ⚠ {len(angostas)} puerta(s) < 0.90 m — verificar con cota en plano (OGUC Art. 4.2.2)")

    # Alerta escaleras angostas (OGUC Art. 4.2.4 — mín 1.20 m)
    esc_angostas = [e for e in elementos if e["tipo"] == "escalera" and e["ancho_m"] < 1.20]
    if esc_angostas:
        print(f"    ⚠ {len(esc_angostas)} escalera(s) < 1.20 m — verificar (OGUC Art. 4.2.4)")

    print()

# Totales globales
puertas_n   = sum(sum(1 for e in r.get("elementos_dino",[]) if e["tipo"]=="puerta")    for r in resultados_paginas)
ventanas_n  = sum(sum(1 for e in r.get("elementos_dino",[]) if e["tipo"]=="ventana")   for r in resultados_paginas)
escaleras_n = sum(sum(1 for e in r.get("elementos_dino",[]) if e["tipo"]=="escalera")  for r in resultados_paginas)
rampas_n    = sum(sum(1 for e in r.get("elementos_dino",[]) if e["tipo"]=="rampa")     for r in resultados_paginas)

print(f"✓ Detección completa — totales:")
print(f"  Puertas   : {puertas_n}")
print(f"  Ventanas  : {ventanas_n}")
print(f"  Escaleras : {escaleras_n}")
print(f"  Rampas    : {rampas_n}")
print("\nEjecuta Celda 5 para ver la visualización con detecciones.")

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 5 — Visualización: OpenCV + detecciones DINO
# ══════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import hsv_to_rgb
import cv2, numpy as np

def color_recinto(i, n):
    rgb = hsv_to_rgb([(i / max(n, 1)) * 0.85, 0.60, 0.95])
    return tuple(int(c * 255) for c in rgb)

# Colores DINO por tipo (BGR para cv2)
COLORES_DINO_BGR = {
    "puerta"           : (255, 144,  30),
    "ventana"          : (209, 206,   0),
    "escalera"         : (  0, 140, 255),
    "rampa"            : (211,   0, 148),
    "columna"          : ( 60,  20, 220),
    "salida_emergencia": ( 34, 139,  34),
}

n_rows = len(viz_pages)
fig, axes = plt.subplots(n_rows, 2, figsize=(22, 11 * n_rows))
if n_rows == 1:
    axes = [axes]

for row, vz in enumerate(viz_pages):
    plano_v    = vz['plano']
    labels_v   = vz['labels']
    recintos_v = vz['recintos_geo']
    ESCALA_V   = vz['escala']
    PAGINA_V   = vz['pagina']
    fname_tag  = vz['fname_tag']
    elementos_dino = vz.get('elementos_dino', [])
    n = len(recintos_v)

    # ── Capa de etiquetas (SIN relleno de color por recinto — desactivado
    # 2026-08-02 a pedido explicito del usuario, el PNG que sube al portal
    # debe quedar sin el color de segmentacion, solo el plano + id/area de
    # cada recinto como texto). anotado = copia limpia del plano, se le
    # agregan solo textos (y bboxes DINO si los hay), nunca color de fondo.
    anotado = plano_v.copy()
    for i, r in enumerate(recintos_v):
        cx, cy = r['cx'], r['cy']
        font   = cv2.FONT_HERSHEY_SIMPLEX
        txts   = [r['id'], f"{r['area_m2']} m2"]
        if r['ancho_min_m']:
            txts.append(f"a:{r['ancho_min_m']}m")
        for ti, txt in enumerate(txts):
            yy = cy - 16 + ti * 18
            for dx, dy in [(-1, -1), (1, -1), (-1, 1), (1, 1)]:
                cv2.putText(anotado, txt, (cx + dx - 20, yy + dy), font, 0.55, (0, 0, 0), 2)
            col = (255, 255, 255) if ti == 0 else (230, 230, 60)
            cv2.putText(anotado, txt, (cx - 20, yy), font, 0.55, col, 1)

    # ── Capa DINO (bboxes de elementos) ────────────────────
    for e in elementos_dino:
        x1, y1, x2, y2 = e['bbox_px']
        color_bgr = COLORES_DINO_BGR.get(e['tipo'], (128, 128, 128))
        anotado_bgr = cv2.cvtColor(anotado, cv2.COLOR_RGB2BGR)
        cv2.rectangle(anotado_bgr, (x1, y1), (x2, y2), color_bgr, 3)
        label_txt = f"{e['tipo']} {e['ancho_m']}m ({e['confianza']:.0%})"
        (tw, th), _ = cv2.getTextSize(label_txt, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(anotado_bgr, (x1, y1 - th - 6), (x1 + tw + 4, y1), color_bgr, -1)
        cv2.putText(anotado_bgr, label_txt,
                    (x1 + 2, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
        anotado = cv2.cvtColor(anotado_bgr, cv2.COLOR_BGR2RGB)

    # PNG individual: {BASENAME}_{fname_tag}.png
    # Ejemplo: archicheck_geometrico_pdv_30jun_1729_pag2-1.png
    fname_pag = f'{BASENAME}_{fname_tag}.png'
    cv2.imwrite(fname_pag, cv2.cvtColor(anotado, cv2.COLOR_RGB2BGR))
    print(f'  ✓ PNG guardado: {fname_pag}')

    ax0, ax1 = axes[row][0], axes[row][1]
    ax0.imshow(plano_v)
    ax0.set_title(f'Página {PAGINA_V} [{fname_tag}] — original', fontsize=12, fontweight='bold')
    ax0.axis('off')

    ax1.imshow(anotado)
    dino_info = f" | DINO: {len(elementos_dino)} elementos" if elementos_dino else ""
    ax1.set_title(
        f'Página {PAGINA_V} [{fname_tag}] — OpenCV ({n} espacios){dino_info} | {ESCALA_V}',
        fontsize=12, fontweight='bold'
    )
    ax1.axis('off')

    patches_cv = [
        mpatches.Patch(
            color=[c / 255 for c in color_recinto(i, n)],
            label=f"{r['id']}: {r['area_m2']} m²" + (f" · a:{r['ancho_min_m']}m" if r['ancho_min_m'] else '')
        )
        for i, r in enumerate(recintos_v[:12])
    ]
    patches_dino = [
        mpatches.Patch(facecolor=[c/255 for c in (30,144,255)],  label='Puerta (DINO)'),
        mpatches.Patch(facecolor=[c/255 for c in (0,206,209)],   label='Ventana (DINO)'),
        mpatches.Patch(facecolor=[c/255 for c in (255,140,0)],   label='Escalera (DINO)'),
        mpatches.Patch(facecolor=[c/255 for c in (148,0,211)],   label='Rampa (DINO)'),
    ]
    ax1.legend(
        handles=patches_cv + patches_dino,
        loc='lower right', fontsize=7, framealpha=0.85, ncol=2,
        title='OpenCV + DINO'
    )

plt.suptitle(f'{NOMBRE_PROYECTO} — Capa 1 Geométrica  |  {pdf_name}',
             fontsize=13, fontweight='bold', y=1.005)
plt.tight_layout()

# PNG combinado: {BASENAME}.png
fname_combined = f'{BASENAME}.png'
plt.savefig(fname_combined, dpi=100, bbox_inches='tight')
plt.show()
print(f'✓ PNG combinado guardado: {fname_combined}')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 6 — Informe en consola + guardar JSON
# ══════════════════════════════════════════════════════════
import json
from datetime import datetime

SEP  = '=' * 66
SEP2 = '-' * 66

print(SEP)
print('  ARCHICHECK — INFORME CAPA 1 GEOMETRICA')
print(SEP)
print(f'  Proyecto: {NOMBRE_PROYECTO}')
print(f'  Archivo : {pdf_name}')
print(f'  Fecha   : {datetime.now().strftime("%d/%m/%Y %H:%M")}')
print(f'  Páginas : {" + ".join(str(p["pagina"]) for p in resultados_paginas)}')

for res in resultados_paginas:
    pag      = res['pagina']
    escala   = res['escala']
    tabla    = res['mediciones_geometricas']
    incs     = res['incumplimientos_geo']
    analisis = res['analisis_semantico']
    dino_els = res.get('elementos_dino', [])

    print(f'\n  {SEP2}')
    print(f'  PAGINA {pag} [{res["fname_tag"]}]  |  {escala}  |  {analisis.get("tipo_plano")} — {analisis.get("uso_del_proyecto")}')
    print(f'  Nivel: {analisis.get("nivel")}')
    print(f'  {SEP2}')

    print(f"  {'ID':<7} {'Nombre':<26} {'Tipo':<12} {'Area m2':>8} {'Ancho m':>8}  Estado")
    print(f"  {'--':<7} {'------':<26} {'----':<12} {'-------':>8} {'-------':>8}  ------")
    for f in tabla:
        st  = 'INCUMPLE' if not f['cumple_geo'] else 'OK'
        aw  = str(f['ancho_min_m']) if f['ancho_min_m'] else '-'
        nom = f['nombre'][:25]
        print(f"  {f['id']:<7} {nom:<26} {f['tipo']:<12} {f['area_m2']:>8.2f} {aw:>8}  {st}")
    print(f"  {'':7} {'TOTAL':26} {'':12} {res['total_area_m2']:>8.2f}")

    if incs:
        print(f'\n  INCUMPLIMIENTOS GEOMETRICOS ({len(incs)})')
        print(f'  {SEP2}')
        for inc in incs:
            tipo_inc = inc['tipo'].upper()
            print(f"  [{tipo_inc}]  {inc['id']} {inc['recinto']}")
            # FIX 2026-07-31: 'discrepancia_area_declarada' (cruce cuadro de
            # superficies vs. area medida, agregado 2026-07-26) usa un
            # esquema de claves distinto ('declarado'/'diff_pct', sin
            # 'minimo'/'deficit') al resto de incumplimientos_geo -- nunca
            # se habia probado con datos reales porque cuadro_superficies_
            # oficial siempre llegaba vacio antes de la funcionalidad de
            # extraccion de texto agregada esta sesion. Primera corrida real
            # (Campo Lindo, 2026-07-31) disparo un KeyError: 'minimo' al
            # asumir el esquema viejo para todos los tipos.
            if inc['tipo'] == 'discrepancia_area_declarada':
                print(f"    Medido: {inc['medido']} m2  |  Declarado (cuadro): {inc['declarado']} m2  |  Diferencia: {inc['diff_pct']}%")
            else:
                unidad = 'm2' if inc['tipo'] == 'area' else 'm'
                print(f"    Medido: {inc['medido']} {unidad}  |  Minimo: {inc['minimo']} {unidad}  |  Deficit: {inc['deficit']} {unidad}")
            print(f"    Ref: {inc['ref']}")

    inc_sem = analisis.get('incumplimientos_oguc', [])
    if inc_sem:
        print(f'\n  OBSERVACIONES NORMATIVAS CLAUDE ({len(inc_sem)})')
        for inc in inc_sem:
            g = inc.get('gravedad', '?')
            d = inc.get('descripcion', '')[:80]
            print(f"  [{g}] {inc.get('articulo', '')} — {d}")

    if dino_els:
        conteo_dino = {}
        for e in dino_els:
            conteo_dino[e['tipo']] = conteo_dino.get(e['tipo'], 0) + 1
        print(f'\n  ELEMENTOS DINO ({len(dino_els)}): {conteo_dino}')
        angostas = [e for e in dino_els if e['tipo'] == 'puerta' and e['ancho_m'] < 0.90]
        esc_ang  = [e for e in dino_els if e['tipo'] == 'escalera' and e['ancho_m'] < 1.20]
        if angostas:
            print(f"    ⚠ {len(angostas)} puerta(s) < 0.90 m (OGUC Art. 4.2.2)")
        if esc_ang:
            print(f"    ⚠ {len(esc_ang)} escalera(s) < 1.20 m (OGUC Art. 4.2.4)")

print(f'\n{SEP}')

# ── Conteos globales DINO ──────────────────────────────────
puertas_n   = sum(sum(1 for e in r.get('elementos_dino',[]) if e['tipo']=='puerta')    for r in resultados_paginas)
ventanas_n  = sum(sum(1 for e in r.get('elementos_dino',[]) if e['tipo']=='ventana')   for r in resultados_paginas)
escaleras_n = sum(sum(1 for e in r.get('elementos_dino',[]) if e['tipo']=='escalera')  for r in resultados_paginas)
rampas_n    = sum(sum(1 for e in r.get('elementos_dino',[]) if e['tipo']=='rampa')     for r in resultados_paginas)

# Fallback: si DINO no detectó nada, usar conteos del análisis semántico Claude
fuente_conteo = 'dino'
if puertas_n == 0 and ventanas_n == 0 and escaleras_n == 0 and rampas_n == 0:
    for r in resultados_paginas:
        sem = r.get('analisis_semantico', {}).get('elementos_detectados', {})
        puertas_n   += sem.get('puertas', 0)
        ventanas_n  += sem.get('ventanas', 0)
        escaleras_n += sem.get('escaleras', 0)
    if puertas_n + ventanas_n + escaleras_n > 0:
        fuente_conteo = 'semantico'
        print(f'  (conteos desde análisis semántico Claude — DINO no detectó elementos)')
        print(f'  Puertas semántico: {puertas_n}  Ventanas: {ventanas_n}  Escaleras: {escaleras_n}')

resultado_final = {
    'fuente'          : 'colab_opencv_multipagina',
    'proyecto'        : NOMBRE_PROYECTO,
    'archivo'         : pdf_name,
    'fecha'           : datetime.now().isoformat(),
    'basename'        : BASENAME,
    'dpi'             : DPI,
    'paginas'         : resultados_paginas,
    'resumen_global'  : {
        'paginas_analizadas'       : len(resultados_paginas),
        'total_area_m2'            : round(sum(p['total_area_m2'] for p in resultados_paginas), 1),
        'total_recintos'           : sum(len(p['mediciones_geometricas']) for p in resultados_paginas),
        'incumplimientos_geo_total': sum(len(p['incumplimientos_geo']) for p in resultados_paginas),
        'puertas_detectadas'       : puertas_n,
        'ventanas_detectadas'      : ventanas_n,
        'escaleras_detectadas'     : escaleras_n,
        'rampas_detectadas'        : rampas_n,
        'fuente_conteo_elementos'  : fuente_conteo,
    }
}

# JSON: {BASENAME}.json → archicheck_geometrico_pdv_30jun_1729.json
fname_json = f'{BASENAME}.json'
with open(fname_json, 'w', encoding='utf-8') as f:
    json.dump(resultado_final, f, ensure_ascii=False, indent=2)

rg = resultado_final['resumen_global']
print(f'\n✓ JSON guardado: {fname_json}')
print(f'  Paginas   : {rg["paginas_analizadas"]}')
print(f'  Area total: {rg["total_area_m2"]} m2')
print(f'  Recintos  : {rg["total_recintos"]}')
print(f'  Incumpl.  : {rg["incumplimientos_geo_total"]}')
print(f'  Elementos ({rg["fuente_conteo_elementos"]}) — Puertas:{rg["puertas_detectadas"]}  Ventanas:{rg["ventanas_detectadas"]}  Escaleras:{rg["escaleras_detectadas"]}  Rampas:{rg["rampas_detectadas"]}')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 7 — Descargar resultados a tu PC
# ══════════════════════════════════════════════════════════
from google.colab import files

print(f'Proyecto: {NOMBRE_PROYECTO}  |  {BASENAME}')
print()

print(f'Descargando JSON → {BASENAME}.json')
files.download(f'{BASENAME}.json')

print(f'Descargando PNG combinado → {BASENAME}.png')
files.download(f'{BASENAME}.png')

print('Descargando PNGs individuales por planta...')
for vz in viz_pages:
    fname = f'{BASENAME}_{vz["fname_tag"]}.png'
    files.download(fname)
    print(f'  → {fname}')

print()
print('✓ Revisa tu carpeta de Descargas.')
print()
print('Siguiente: en ArchiCheck web sube el PDF + el JSON')
print('+ los PNG de cada planta como archivos adicionales.')